# 🌊 JAWALGAON DAM BREAK — Complete PAR Analysis
## Emergency Action Plan · GIS Assessment · Scenarios: Piping | Overtopping
---
**Run cells top to bottom. Every cell is self-contained and sequential.**

| Cell | Task |
|---|---|
| 1 | Install libraries |
| 2 | Configuration — edit file names & column names here |
| 3 | Validate all input files |
| 4 | Load shapefiles + auto-detect UTM CRS |
| 5 | Inspect columns (run before editing Cell 2) |
| 6 | Polygon smoothing — fix jagged digitization |
| 7 | Dam location detection |
| 8 | Reproject all rasters to UTM |
| 9 | Generate inundation polygons |
| 10 | Reclassify arrival time (T=0 at breach) |
| 11 | H1–H6 hazard classification rasters |
| 12 | OSM bulk query — roads + all POI in one request |
| 13 | Zonal statistics per settlement (depth/velocity/WSE/arrival) |
| 14 | PAR calculation + flood status callouts |
| 15 | Evacuation priority + vulnerability index |
| 16 | Nearest shelter assignment |
| 17 | Road + infrastructure inundation analysis |
| 18 | Save all shapefiles |
| 19 | Build Annexure 3A & 3B tables |
| 20 | Full colour-coded visualizations in Colab |
| 21 | Summary report |
| 22 | ZIP & download |

## Cell 1 — Install Libraries

In [ ]:
!pip install geopandas rasterio rasterstats shapely fiona pyproj requests numpy pandas openpyxl scipy --quiet
print("✅ Libraries installed.")

✅ Libraries installed.


## Cell 2 — Configuration
> ✏️ **Only edit values in this cell. Run Cell 5 first to confirm column names.**

In [ ]:
# Cell 2
import os, warnings, time, requests
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import shapes as rio_shapes
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterstats import zonal_stats
from shapely.geometry import Point, LineString, Polygon, MultiPolygon, shape
from shapely.ops import unary_union
from scipy.ndimage import gaussian_filter
import shutil
from IPython.display import display, HTML
warnings.filterwarnings("ignore")

# ── Project ──────────────────────────────────────────────────────────────────
DAM_NAME     = "Jawalgaon Dam"
DAM_DISTRICT = "Solapur"
DAM_STATE    = "Maharashtra"

# ── Directories ───────────────────────────────────────────────────────────────
BASE    = "/content"
OUT_DIR = os.path.join(BASE, "PAR_Outputs")
os.makedirs(OUT_DIR, exist_ok=True)
REPROJ_DIR = os.path.join(OUT_DIR, "_reproj")
os.makedirs(REPROJ_DIR, exist_ok=True)

# ── Settlement column names — edit to match your shapefile ───────────────────
NAME_COLUMN       = "Name"          # village name field
POP_COLUMN        = "Pop"           # total population field
POP_MALE_COLUMN   = None            # male pop field (None if absent)
POP_FEMALE_COLUMN = None            # female pop field (None if absent)
HH_COLUMN         = None            # household count (None if absent)
SHELTER_NAME_COL  = "Shelter"       # shelter name field
SHELTER_CAP_COL   = None            # shelter capacity (None if absent)

# ── Arrival time offset ───────────────────────────────────────────────────────
OVT_OFFSET_HRS = 19.0   # overtopping simulation time when flood starts at dam
# piping offset = auto-detected from raster minimum

# ── Rasters ───────────────────────────────────────────────────────────────────
PIPING = {
    "depth":        os.path.join(BASE, "Depth (Max).Terrain.SENTINAL-30M-DEM.tif"),
    "velocity":     os.path.join(BASE, "Velocity (Max).Terrain.SENTINAL-30M-DEM.tif"),
    "arrival_time": os.path.join(BASE, "Arrival Time (Max).Terrain.SENTINAL-30M-DEM.tif"),
    "wse":          os.path.join(BASE, "WSE (Max).Terrain.SENTINAL-30M-DEM.tif"),
}
OVERTOPPING = {
    "depth":        os.path.join(BASE, "Depth (Max).SENTINAL-30M-DEM.SENTINAL-30M-DEM.tif"),
    "velocity":     os.path.join(BASE, "Velocity (Max).SENTINAL-30M-DEM.SENTINAL-30M-DEM.tif"),
    "arrival_time": os.path.join(BASE, "Arrival Time (Max).SENTINAL-30M-DEM.SENTINAL-30M-DEM.tif"),
    "wse":          os.path.join(BASE, "WSE (Max).SENTINAL-30M-DEM.SENTINAL-30M-DEM.tif"),
}

# ── Shapefiles ────────────────────────────────────────────────────────────────
SHP_SETTLEMENTS = os.path.join(BASE, "village_polygon.shp")
SHP_RIVER       = os.path.join(BASE, "River.shp")
SHP_RESERVOIR   = os.path.join(BASE, "Submurgen.shp")
SHP_WATERSHED   = os.path.join(BASE, "Watershed.shp")
SHP_SHELTERS    = os.path.join(BASE, "shelter locations_Jawalgaon dam failure.shp")

# ── Hazard thresholds — DV product (m²/s) ────────────────────────────────────
HAZARD_CFG = {
    "H1": {"lo": 0.00, "hi": 0.25, "label": "Very Low",  "code": 1},
    "H2": {"lo": 0.25, "hi": 0.50, "label": "Low",       "code": 2},
    "H3": {"lo": 0.50, "hi": 1.00, "label": "Medium",    "code": 3},
    "H4": {"lo": 1.00, "hi": 2.00, "label": "High",      "code": 4},
    "H5": {"lo": 2.00, "hi": 4.00, "label": "Very High", "code": 5},
    "H6": {"lo": 4.00, "hi": 9999, "label": "Extreme",   "code": 6},
}

# ── Flood status callouts ─────────────────────────────────────────────────────
def flood_status_label(pct, surrounded=False, isolated=False):
    if surrounded:  return "VILLAGE SURROUNDED BY FLOOD"
    if isolated:    return "VILLAGE ISOLATED — ROAD ACCESS CUT"
    if pct >= 95:   return "VILLAGE COMPLETELY FLOODED"
    if pct >= 70:   return "VILLAGE HEAVILY FLOODED"
    if pct >= 30:   return "VILLAGE MODERATELY FLOODED"
    if pct >=  5:   return "VILLAGE PARTIALLY FLOODED"
    if pct >   0:   return "VILLAGE MARGINALLY AFFECTED"
    return "NOT AFFECTED"

# ── Evacuation priority — relative arrival time (hrs after breach) ────────────
def evac_priority(arr_hrs):
    """Returns (order, window, code, label)."""
    if arr_hrs is None: return (5, "8+ hours", "P5", "PLANNED")
    try:
        t = float(arr_hrs)
    except (TypeError, ValueError):
        return (5, "8+ hours", "P5", "PLANNED")
    if t >= 9999:   return (5, "8+ hours",    "P5", "PLANNED")
    if t <= 2:      return (1, "0 to 2 hour", "P1", "IMMEDIATE")
    if t <= 4:      return (2, "2–4 hours",   "P2", "URGENT")
    if t <= 6:      return (3, "4–6 hours",   "P3", "HIGH")
    if t <= 8:      return (4, "6–8 hours",   "P4", "MODERATE")
    return                 (5, "8+ hours",    "P5", "PLANNED")

# ── Road category map ─────────────────────────────────────────────────────────
ROAD_CAT = {
    "motorway":"National Highway", "trunk":"State Highway",
    "primary":"Major Road",        "secondary":"District Road",
    "tertiary":"Village Road",     "unclassified":"Village Road",
    "residential":"Internal Road", "service":"Service Road",
    "track":"Kutcha Track",        "path":"Footpath",
    "footway":"Footpath",          "motorway_link":"Highway Link",
    "trunk_link":"Highway Link",   "primary_link":"Road Link",
    "secondary_link":"Road Link",
}

print(f"✅ Configuration ready — {DAM_NAME}, {DAM_DISTRICT}, {DAM_STATE}")
print(f"   Output folder: {OUT_DIR}")

✅ Configuration ready — Jawalgaon Dam, Solapur, Maharashtra
   Output folder: /content/PAR_Outputs


## Cell 3 — Validate Input Files

In [ ]:
# Cell 3
all_files = list(PIPING.values()) + list(OVERTOPPING.values()) + [
    SHP_SETTLEMENTS, SHP_RIVER, SHP_RESERVOIR, SHP_WATERSHED, SHP_SHELTERS
]
missing = [f for f in all_files if not os.path.exists(f)]
if missing:
    print("❌ MISSING FILES — upload to /content/ then re-run:")
    for f in missing:
        print(f"   → {os.path.basename(f)}")
else:
    print("✅ All input files found!")

✅ All input files found!


## Cell 4 — Load Shapefiles & Auto-detect UTM CRS

In [ ]:
# Cell 4
def detect_utm(gdf):
    g   = gdf.to_crs("EPSG:4326")
    lon = g.geometry.centroid.x.mean()
    lat = g.geometry.centroid.y.mean()
    z   = int((lon + 180) / 6) + 1
    return f"EPSG:{32600 + z if lat >= 0 else 32700 + z}"

def load_shp(path, crs):
    return gpd.read_file(path).to_crs(crs)

watershed   = gpd.read_file(SHP_WATERSHED)
TARGET_CRS  = detect_utm(watershed)
watershed   = watershed.to_crs(TARGET_CRS)
BBOX_WGS84  = watershed.to_crs("EPSG:4326").total_bounds  # [minx,miny,maxx,maxy]

settlements_raw = load_shp(SHP_SETTLEMENTS, TARGET_CRS)
river           = load_shp(SHP_RIVER,       TARGET_CRS)
reservoir       = load_shp(SHP_RESERVOIR,   TARGET_CRS)
shelters        = load_shp(SHP_SHELTERS,    TARGET_CRS)

sett_ply = settlements_raw[
    settlements_raw.geometry.geom_type.isin(["Polygon","MultiPolygon"])
].copy().reset_index(drop=True)

sett_pt = settlements_raw[
    settlements_raw.geometry.geom_type == "Point"
].copy().reset_index(drop=True)

print(f"✅ CRS: {TARGET_CRS}")
print(f"   Settlement polygons : {len(sett_ply)}")
print(f"   Settlement points   : {len(sett_pt)}")
print(f"   Shelters            : {len(shelters)}")
print(f"   Bounding box (WGS84): {BBOX_WGS84.round(4)}")

✅ CRS: EPSG:32643
   Settlement polygons : 22
   Settlement points   : 0
   Shelters            : 16
   Bounding box (WGS84): [75.7282 17.9497 76.0992 18.108 ]


## Cell 5 — Inspect Columns (run before editing Cell 2)

In [ ]:
# Cell 5
print("=== SETTLEMENT COLUMNS ===")
print(settlements_raw.dtypes.to_string())
print()
display(settlements_raw.drop(columns="geometry").head(3))
print("\n=== SHELTER COLUMNS ===")
print(shelters.dtypes.to_string())
display(shelters.drop(columns="geometry").head(3))

=== SETTLEMENT COLUMNS ===
Name          object
Pop           object
geometry    geometry



,Name,Pop
0,Mungshi,1291
1,Dahithane,1765
2,Sasure,3427



=== SHELTER COLUMNS ===
id             int64
Shelter       object
geometry    geometry


,id,Shelter
0,1,Shelter for Hatij Flood Afffected People
1,2,None
2,3,None


## Cell 6 — Polygon Smoothing (Chaikin corner-cutting)
> Fixes over-sharp jagged digitization. Slightly expands area for more realistic coverage.

In [ ]:
# Cell 6
def chaikin(coords, iters=4):
    pts = list(coords)
    if pts[0] != pts[-1]:
        pts.append(pts[0])
    for _ in range(iters):
        new = []
        for i in range(len(pts) - 1):
            x0, y0 = pts[i];   x1, y1 = pts[i+1]
            new.append((0.75*x0 + 0.25*x1, 0.75*y0 + 0.25*y1))
            new.append((0.25*x0 + 0.75*x1, 0.25*y0 + 0.75*y1))
        new.append(new[0])
        pts = new
    return pts

def smooth_geom(geom, iters=4, buf=15, tol=5):
    if geom is None or geom.is_empty:
        return geom
    if geom.geom_type == "MultiPolygon":
        parts = [smooth_geom(p, iters, buf, tol) for p in geom.geoms]
        return MultiPolygon([p for p in parts if p and not p.is_empty])
    try:
        g = geom.buffer(buf, join_style=1).buffer(-buf * 0.5, join_style=1)
        ext = chaikin(list(g.exterior.coords), iters)
        if len(ext) < 4:
            return g
        s = Polygon(ext).simplify(tol, preserve_topology=True)
        return s if s.is_valid else g
    except Exception:
        return geom

if not sett_ply.empty:
    print("🔄 Smoothing settlement polygons...")
    sett_ply_smooth = sett_ply.copy()
    sett_ply_smooth["geometry"] = sett_ply_smooth["geometry"].apply(smooth_geom)
    delta = ((sett_ply_smooth.area - sett_ply.area) / sett_ply.area * 100).round(1)
    print(f"   Mean area change : +{delta.mean():.1f}%  |  Max: +{delta.max():.1f}%")
    sett_ply_smooth.to_file(os.path.join(OUT_DIR, "settlements_smoothed.shp"))
    print("✅ Saved: settlements_smoothed.shp")
else:
    sett_ply_smooth = sett_ply.copy()
    print("ℹ️  No polygon settlements found.")

🔄 Smoothing settlement polygons...
   Mean area change : +9.9%  |  Max: +16.1%
✅ Saved: settlements_smoothed.shp


## Cell 7 — Dam Location Detection

In [ ]:
# Cell 7
def get_dam_point(res_gdf, riv_gdf):
    res_u = res_gdf.geometry.unary_union
    riv_u = riv_gdf.geometry.unary_union
    coords = ([c for l in riv_u.geoms for c in l.coords]
              if hasattr(riv_u, "geoms") else list(riv_u.coords))
    pts = [Point(c) for c in coords]
    return min(pts, key=lambda p: res_u.boundary.distance(p))

dam_point = get_dam_point(reservoir, river)
dam_gdf   = gpd.GeoDataFrame(
    {"dam_name": [DAM_NAME], "district": [DAM_DISTRICT]},
    geometry=[dam_point], crs=TARGET_CRS)

dam_wgs84 = dam_gdf.to_crs("EPSG:4326").geometry.iloc[0]
print(f"✅ {DAM_NAME} located")
print(f"   Lat: {dam_wgs84.y:.4f}°N  |  Lon: {dam_wgs84.x:.4f}°E")
print(f"   UTM: X={dam_point.x:.1f}  Y={dam_point.y:.1f}")

# River chainage function — used later
def river_chainage(gdf, riv_gdf, dam_pt):
    """Distance (km) from dam along river for each feature."""
    riv  = riv_gdf.geometry.unary_union
    line = max(riv.geoms, key=lambda l: l.length) if hasattr(riv, "geoms") else riv
    d0   = line.project(dam_pt)
    out  = []
    for geom in gdf.geometry:
        pt = geom if geom.geom_type == "Point" else geom.centroid
        out.append(round(abs(line.project(pt) - d0) / 1000, 3))
    return out

✅ Jawalgaon Dam located
   Lat: 18.0171°N  |  Lon: 75.9010°E
   UTM: X=595376.4  Y=1992308.5


## Cell 8 — Reproject All Rasters to Target UTM

In [ ]:
# Cell 8
def reproject_raster(src_path, dst_path, dst_crs):
    if os.path.exists(dst_path):
        return dst_path
    with rasterio.open(src_path) as s:
        t, w, h = calculate_default_transform(
            s.crs, dst_crs, s.width, s.height, *s.bounds)
        meta = s.meta.copy()
        meta.update(crs=dst_crs, transform=t, width=w, height=h)
        with rasterio.open(dst_path, "w", **meta) as d:
            for i in range(1, s.count + 1):
                reproject(rasterio.band(s, i), rasterio.band(d, i),
                          src_transform=s.transform, src_crs=s.crs,
                          dst_transform=t, dst_crs=dst_crs,
                          resampling=Resampling.bilinear)
    return dst_path

def reproj_scenario(d, label):
    out = {}
    for k, p in d.items():
        dp = os.path.join(REPROJ_DIR, f"{label}_{k}.tif")
        out[k] = reproject_raster(p, dp, TARGET_CRS)
        print(f"   ✅ [{label}] {k}")
    return out

print("🔄 Piping rasters...")
PIP_R = reproj_scenario(PIPING, "pip")
print("🔄 Overtopping rasters...")
OVT_R = reproj_scenario(OVERTOPPING, "ovt")
print("✅ All rasters reprojected.")

🔄 Piping rasters...
   ✅ [pip] depth
   ✅ [pip] velocity
   ✅ [pip] arrival_time
   ✅ [pip] wse
🔄 Overtopping rasters...
   ✅ [ovt] depth
   ✅ [ovt] velocity
   ✅ [ovt] arrival_time
   ✅ [ovt] wse
✅ All rasters reprojected.


## Cell 9 — Generate Inundation Polygons

In [ ]:
# Cell 9
def depth_to_poly(depth_tif, min_d=0.01):
    with rasterio.open(depth_tif) as s:
        data, nd, tf, crs = s.read(1), s.nodata, s.transform, s.crs
    mask = data > min_d
    if nd is not None:
        mask &= (data != nd)
    polys = [shape(g) for g, v in
             rio_shapes(mask.astype("uint8"), mask=mask.astype("uint8"), transform=tf)
             if v == 1]
    if not polys:
        raise ValueError("No inundated pixels found. Check raster values.")
    return gpd.GeoDataFrame(geometry=[unary_union(polys)], crs=crs).to_crs(TARGET_CRS)

print("🔄 Building inundation polygons...")
inund_pip = depth_to_poly(PIP_R["depth"])
inund_ovt = depth_to_poly(OVT_R["depth"])

area_pip_ha = inund_pip.geometry.area.sum() / 10000
area_ovt_ha = inund_ovt.geometry.area.sum() / 10000

print(f"✅ Piping inundation area      : {area_pip_ha:,.2f} ha  ({area_pip_ha/100:.3f} km²)")
print(f"✅ Overtopping inundation area : {area_ovt_ha:,.2f} ha  ({area_ovt_ha/100:.3f} km²)")

inund_pip.to_file(os.path.join(OUT_DIR, "inundation_piping.shp"))
inund_ovt.to_file(os.path.join(OUT_DIR, "inundation_overtopping.shp"))
dam_gdf.to_file(os.path.join(OUT_DIR, "dam_location.shp"))
print("✅ Inundation polygons saved.")

🔄 Building inundation polygons...
✅ Piping inundation area      : 2,152.54 ha  (21.525 km²)
✅ Overtopping inundation area : 3,240.84 ha  (32.408 km²)
✅ Inundation polygons saved.


## Cell 10 — Arrival Time Reclassification
> Subtracts simulation start time so T=0 means dam breach moment.

In [ ]:
# Cell 10
def reclassify_arrival(tif_path, offset_hrs, out_path):
    with rasterio.open(tif_path) as s:
        data, nd, profile = s.read(1).astype(float), s.nodata, s.profile.copy()
    nd_mask = ((data == nd) | np.isnan(data)) if nd is not None else np.isnan(data)
    valid   = data[~nd_mask]
    raw_min = float(valid.min()) if valid.size > 0 else offset_hrs
    raw_max = float(valid.max()) if valid.size > 0 else offset_hrs
    fill_nd = nd if nd is not None else -9999
    rel     = np.where(nd_mask, fill_nd, np.maximum(0.0, data - offset_hrs))
    profile.update(dtype="float32", compress="lzw")
    if nd is None:
        profile["nodata"] = -9999
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(rel.astype("float32"), 1)
    rel_valid = rel[~nd_mask]
    print(f"   Raw range  : {raw_min:.2f}–{raw_max:.2f} hrs  |  Offset: {offset_hrs:.2f} hrs")
    print(f"   Rel range  : {rel_valid.min():.2f}–{rel_valid.max():.2f} hrs")
    return out_path

print("🔄 Reclassifying Overtopping arrival time (offset = 19 hrs)...")
OVT_ARR_REL = reclassify_arrival(
    OVT_R["arrival_time"], OVT_OFFSET_HRS,
    os.path.join(OUT_DIR, "arrival_overtopping_relative.tif"))

print("\n🔄 Reclassifying Piping arrival time (auto-detect offset)...")
with rasterio.open(PIP_R["arrival_time"]) as s:
    raw = s.read(1).astype(float)
    nd  = s.nodata
    v   = raw[raw != nd] if nd else raw[~np.isnan(raw)]
    pip_offset = float(v.min()) if v.size > 0 else 0.0
print(f"   Piping offset auto-detected: {pip_offset:.2f} hrs")
PIP_ARR_REL = reclassify_arrival(
    PIP_R["arrival_time"], pip_offset,
    os.path.join(OUT_DIR, "arrival_piping_relative.tif"))

print("\n✅ Arrival time reclassification done.")

🔄 Reclassifying Overtopping arrival time (offset = 19 hrs)...


   Raw range  : 16.18–70.37 hrs  |  Offset: 19.00 hrs
   Rel range  : 0.00–51.37 hrs

🔄 Reclassifying Piping arrival time (auto-detect offset)...
   Piping offset auto-detected: 0.29 hrs


   Raw range  : 0.29–65.01 hrs  |  Offset: 0.29 hrs
   Rel range  : 0.00–64.72 hrs

✅ Arrival time reclassification done.


## Cell 11 — H1–H6 Hazard Classification Rasters
> DV = Depth (m) × Velocity (m/s). H1=Very Low … H6=Extreme.

In [ ]:
# Cell 11
def make_hazard_raster(depth_tif, vel_tif, out_path):
    with rasterio.open(depth_tif) as ds:
        depth, nd_d, profile = ds.read(1).astype(float), ds.nodata, ds.profile.copy()
    with rasterio.open(vel_tif) as vs:
        vel, nd_v = vs.read(1).astype(float), vs.nodata
    if nd_d: depth[depth == nd_d] = 0
    if nd_v: vel[vel == nd_v]     = 0
    depth = np.clip(depth, 0, None)
    vel   = np.clip(vel,   0, None)
    dv    = depth * vel
    hz    = np.zeros_like(dv, dtype="uint8")
    inund = depth > 0.01
    for cls, cfg in HAZARD_CFG.items():
        hz[inund & (dv >= cfg["lo"]) & (dv < cfg["hi"])] = cfg["code"]
    profile.update(dtype="uint8", count=1, nodata=0, compress="lzw")
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(hz, 1)
    tot = int((hz > 0).sum())
    for cls, cfg in HAZARD_CFG.items():
        cnt = int((hz == cfg["code"]).sum())
        pct = cnt / tot * 100 if tot > 0 else 0
        print(f"   {cls} ({cfg['label']:<10}): {cnt:>8,} px  ({pct:5.1f}%)")
    return out_path

HZ_PIP_TIF = os.path.join(OUT_DIR, "hazard_piping.tif")
HZ_OVT_TIF = os.path.join(OUT_DIR, "hazard_overtopping.tif")

print("🔄 Piping hazard raster:")
make_hazard_raster(PIP_R["depth"], PIP_R["velocity"], HZ_PIP_TIF)
print("\n🔄 Overtopping hazard raster:")
make_hazard_raster(OVT_R["depth"], OVT_R["velocity"], HZ_OVT_TIF)
print("\n✅ Hazard rasters saved.")

🔄 Piping hazard raster:


   H1 (Very Low  ):  105,289 px  ( 48.9%)
   H2 (Low       ):   15,458 px  (  7.2%)
   H3 (Medium    ):   19,970 px  (  9.3%)
   H4 (High      ):   27,472 px  ( 12.8%)
   H5 (Very High ):   24,158 px  ( 11.2%)
   H6 (Extreme   ):   22,907 px  ( 10.6%)

🔄 Overtopping hazard raster:


   H1 (Very Low  ):  125,303 px  ( 38.7%)
   H2 (Low       ):   16,284 px  (  5.0%)
   H3 (Medium    ):   21,362 px  (  6.6%)
   H4 (High      ):   32,134 px  (  9.9%)
   H5 (Very High ):   39,304 px  ( 12.1%)
   H6 (Extreme   ):   89,697 px  ( 27.7%)

✅ Hazard rasters saved.


## Cell 12 — OSM Bulk Query (single API call — avoids 429)
> Roads + all POI + Bridges/Canals/Barrages in **two** HTTP requests total.

In [ ]:
# Cell 12
def overpass_fetch(query, max_retries=6):
    url  = "https://overpass-api.de/api/interpreter"
    wait = 15
    for attempt in range(1, max_retries + 1):
        try:
            print(f"   📡 Attempt {attempt}...")
            r = requests.post(url, data={"data": query}, timeout=360)
            if r.status_code in (429, 504):
                print(f"   ⚠️  HTTP {r.status_code} — waiting {wait}s...")
                time.sleep(wait); wait = min(wait * 2, 120); continue
            r.raise_for_status()
            elems = r.json().get("elements", [])
            print(f"   ✅ {len(elems)} elements received")
            return r.json()
        except requests.exceptions.Timeout:
            print(f"   ⚠️  Timeout — waiting {wait}s...")
            time.sleep(wait); wait = min(wait * 2, 120)
        except Exception as e:
            print(f"   ❌ {e}")
            if attempt < max_retries: time.sleep(wait)
    return {"elements": []}

minx, miny, maxx, maxy = BBOX_WGS84
bb = f"{miny},{minx},{maxy},{maxx}"

# ── All POI tags → category ───────────────────────────────────────────────────
TAG_CAT = {
    "amenity=townhall":            "Grampanchayat/Municipal",
    "office=government":           "Grampanchayat/Municipal",
    "office=administrative":       "Grampanchayat/Municipal",
    "amenity=public_building":     "Grampanchayat/Municipal",
    "amenity=hospital":            "Hospital",
    "amenity=clinic":              "PHC/Clinic",
    "amenity=health_centre":       "PHC/Clinic",
    "healthcare=centre":           "PHC/Clinic",
    "amenity=doctors":             "PHC/Clinic",
    "amenity=pharmacy":            "Pharmacy",
    "amenity=school":              "School",
    "amenity=college":             "College",
    "amenity=kindergarten":        "Anganwadi/Kindergarten",
    "amenity=social_facility":     "Anganwadi/Social Centre",
    "amenity=place_of_worship":    "Temple/Religious",
    "amenity=police":              "Police Station",
    "amenity=fire_station":        "Fire Station",
    "amenity=bank":                "Bank",
    "amenity=atm":                 "ATM",
    "amenity=post_office":         "Post Office",
    "highway=bus_stop":            "Bus Stop",
    "amenity=bus_station":         "Bus Station",
    "amenity=marketplace":         "Market/Bazaar",
    "man_made=water_well":         "Water Well",
    "man_made=water_tower":        "Water Tower/Tank",
    "power=substation":            "Power Substation",
    "amenity=community_centre":    "Community Centre",
    "amenity=shelter":             "Shelter",
    # Hydraulic infrastructure
    "bridge=yes":                  "Bridge",
    "man_made=bridge":             "Bridge",
    "waterway=canal":              "Canal",
    "waterway=weir":               "Weir",
    "man_made=barrage":            "Barrage",
    "waterway=dam":                "Dam",
}

def build_poi_query(bb, tag_cat):
    lines = []
    for tag in tag_cat:
        k, v = tag.split("=", 1)
        lines.append(f'  node["{k}"="{v}"]({bb});')
        lines.append(f'  way["{k}"="{v}"]({bb});')
    return f"[out:json][timeout:300];\n(\n{''.join(lines)}\n);\nout center;"

def get_category(tags):
    for tag, cat in TAG_CAT.items():
        k, v = tag.split("=", 1)
        if tags.get(k) == v:
            return cat
    return "Other"

def fill_name(row_dict):
    nm = str(row_dict.get("name", "")).strip()
    if nm in ("", "None", "nan"):
        cat = str(row_dict.get("category", "Feature"))
        return f"Unnamed {cat} (OSM:{row_dict.get('osm_id', '?')})"
    return nm

print("🌐 Fetching POI + hydraulic infrastructure (bulk query)...")
poi_data = overpass_fetch(build_poi_query(bb, TAG_CAT))

poi_rows = []
seen_ids = set()
for el in poi_data.get("elements", []):
    eid = el.get("id")
    if eid in seen_ids: continue
    seen_ids.add(eid)
    if el["type"] == "node":
        lat, lon = el.get("lat"), el.get("lon")
    elif el["type"] == "way" and "center" in el:
        lat, lon = el["center"]["lat"], el["center"]["lon"]
    else:
        continue
    if lat is None or lon is None: continue
    t = el.get("tags", {})
    d = {"osm_id": eid, "category": get_category(t),
         "name":    t.get("name", t.get("name:en", "")),
         "name_mr": t.get("name:mr", ""),
         "amenity": t.get("amenity", ""),
         "office":  t.get("office", ""),
         "operator":t.get("operator", ""),
         "phone":   t.get("phone", t.get("contact:phone", "")),
         "addr":    t.get("addr:full", t.get("addr:village", "")),
         "geometry":Point(lon, lat)}
    d["name"] = fill_name(d)
    poi_rows.append(d)

osm_poi = (gpd.GeoDataFrame(poi_rows, crs="EPSG:4326").to_crs(TARGET_CRS)
           if poi_rows else gpd.GeoDataFrame())

# ══════════════════════════════════════════════════════════════
#  ROAD NAME FIX — replace the road_rows loop in Cell 12
#  Extracts: ref (NH/SH), name, name:mr, name:hi, destination
#  Builds a readable label in priority order
# ══════════════════════════════════════════════════════════════

road_rows = []
for el in road_data.get("elements", []):
    if el["type"] != "way" or "geometry" not in el:
        continue
    coords = [(n["lon"], n["lat"]) for n in el["geometry"]]
    if len(coords) < 2:
        continue

    t  = el.get("tags", {})
    hw = t.get("highway", "unknown")

    # ── Extract every useful name/reference tag ─────────────────
    ref         = t.get("ref", "")           # NH 65, SH 150, MDR 12
    name_en     = t.get("name", "") or t.get("name:en", "")
    name_mr     = t.get("name:mr", "")       # Marathi name
    name_hi     = t.get("name:hi", "")       # Hindi name
    loc_name    = t.get("loc_name", "")      # local colloquial name
    destination = t.get("destination", "")   # e.g. "Solapur;Akkalkot"
    dest_ref    = t.get("destination:ref","")# destination road ref

    # ── Build display name in priority order ─────────────────────
    # Priority: ref > english name > marathi name > hindi name >
    #           local name > destination > road category fallback
    if ref and name_en:
        # Best case: "NH 65 — Solapur–Akkalkot Road"
        display_name = f"{ref} — {name_en}"
    elif ref and destination:
        # "SH 150 → Akkalkot"
        dest_clean = destination.replace(";", " → ")
        display_name = f"{ref} → {dest_clean}"
    elif ref:
        # Just the reference number
        display_name = ref
    elif name_en:
        display_name = name_en
    elif name_mr:
        display_name = name_mr          # Marathi script name
    elif name_hi:
        display_name = name_hi
    elif loc_name:
        display_name = loc_name
    elif destination:
        dest_clean   = destination.replace(";", " → ")
        road_type_lbl = ROAD_CAT.get(hw, "Road")
        display_name  = f"{road_type_lbl} → {dest_clean}"
    else:
        # Truly unnamed — rural tracks often have no name in OSM
        # Show road type + OSM ID so it's traceable
        display_name = f"Unnamed {ROAD_CAT.get(hw,'Road')} (OSM:{el['id']})"

    road_rows.append({
        "osm_id":    el["id"],
        "name":      display_name,          # ← the clean label used in tables
        "ref":       ref,                   # raw ref for filtering (NH/SH)
        "name_mr":   name_mr,               # Marathi name column
        "road_type": hw,                    # primary / track / etc.
        "road_cat":  ROAD_CAT.get(hw, "Other Road"),
        "surface":   t.get("surface", ""),
        "geometry":  LineString(coords),
    })

# Rebuild GDF
if road_rows:
    osm_roads = gpd.GeoDataFrame(road_rows, crs="EPSG:4326").to_crs(TARGET_CRS)
    print(f"✅ {len(osm_roads)} road segments loaded")

    # Show how many got real names vs unnamed
    named   = osm_roads[~osm_roads["name"].str.startswith("Unnamed")]["name"]
    unnamed = osm_roads[osm_roads["name"].str.startswith("Unnamed")]
    print(f"   Named roads    : {len(named)}")
    print(f"   Unnamed roads  : {len(unnamed)}  (no OSM name — common in rural areas)")

    # Show sample of named roads
    if len(named) > 0:
        print("\n   Sample named roads:")
        for nm in named.head(10).values:
            print(f"     • {nm}")
else:
    osm_roads = gpd.GeoDataFrame()
    print("⚠️  No roads fetched.")

🌐 Fetching POI + hydraulic infrastructure (bulk query)...
   📡 Attempt 1...
   ✅ 361 elements received
✅ 2653 road segments loaded
   Named roads    : 150
   Unnamed roads  : 2503  (no OSM name — common in rural areas)

   Sample named roads:
     • SH151
     • SH151
     • SH150
     • SH150
     • SH153
     • SH150
     • NH652
     • SH150
     • SH150
     • SH153


## Cell 13 — Zonal Statistics per Settlement
> Single canonical function. Column names: `pip_dep_max`, `pip_dep_avg`,
> `pip_vel_max`, `pip_vel_avg`, `pip_wse_max`, `pip_wse_avg`,
> `pip_arr_min`, `pip_arr_avg` — same prefix pattern for `ovt_`.

In [ ]:
# Cell 13
def run_zonal_stats(gdf, pip_r, ovt_r, pip_arr, ovt_arr, hz_pip, hz_ovt):
    """
    Extracts all raster parameters per settlement feature.
    Uses reprojected rasters (pip_r/ovt_r) and reclassified arrival rasters.
    All column names use full suffixes: _max, _avg, _min — no abbreviations.
    """
    result = gdf.copy()
    is_pt  = not gdf.empty and gdf.geometry.geom_type.iloc[0] == "Point"
    g      = gdf.geometry.buffer(30) if is_pt else gdf.geometry

    # (param_key, pip_path, ovt_path, stats_list)
    PARAMS = [
        ("dep", pip_r["depth"],    ovt_r["depth"],    ["max", "mean"]),
        ("vel", pip_r["velocity"], ovt_r["velocity"], ["max", "mean"]),
        ("wse", pip_r["wse"],      ovt_r["wse"],      ["max", "mean"]),
        ("arr", pip_arr,           ovt_arr,            ["min", "mean"]),
    ]
    STAT_SUFFIX = {"max": "max", "mean": "avg", "min": "min"}

    for param, pip_p, ovt_p, stats in PARAMS:
        for sc, path in [("pip", pip_p), ("ovt", ovt_p)]:
            try:
                zs = zonal_stats(g, path, stats=stats, nodata=-9999, all_touched=True)
                for s in stats:
                    col  = f"{sc}_{param}_{STAT_SUFFIX[s]}"
                    vals = []
                    for z in zs:
                        v = z.get(s)
                        if v is None or (isinstance(v, float) and np.isnan(v)):
                            vals.append(np.nan)
                        else:
                            vals.append(round(float(v), 4))
                    result[col] = vals
            except Exception as e:
                print(f"   ⚠️  [{sc}] {param} failed: {e}")

    # Hazard class (dominant pixel per polygon)
    for sc, hz_tif in [("pip", hz_pip), ("ovt", hz_ovt)]:
        try:
            zs_hz = zonal_stats(g, hz_tif, stats=["majority"], nodata=0, all_touched=True)
            codes = [int(z.get("majority") or 0) for z in zs_hz]
            result[f"{sc}_hz_cod"] = codes
            result[f"{sc}_hz_cls"] = [
                list(HAZARD_CFG.keys())[c-1] if 1 <= c <= 6 else "None"
                for c in codes
            ]
            result[f"{sc}_hz_lbl"] = [
                HAZARD_CFG[cls]["label"] if cls != "None" else "Not Inundated"
                for cls in result[f"{sc}_hz_cls"]
            ]
        except Exception as e:
            print(f"   ⚠️  [{sc}] hazard class failed: {e}")

    # Distance from dam along river
    for sc in ["pip", "ovt"]:
        result[f"{sc}_dist_km"] = river_chainage(result, river, dam_point)

    return result


print("🔄 Running zonal statistics...")

if not sett_ply_smooth.empty:
    print("   Polygon settlements — Piping...")
    ply_pip = run_zonal_stats(sett_ply_smooth,
                              PIP_R, OVT_R, PIP_ARR_REL, OVT_ARR_REL,
                              HZ_PIP_TIF, HZ_OVT_TIF)
    print("   Polygon settlements — Overtopping...")
    ply_ovt = run_zonal_stats(sett_ply_smooth,
                              PIP_R, OVT_R, PIP_ARR_REL, OVT_ARR_REL,
                              HZ_PIP_TIF, HZ_OVT_TIF)

if not sett_pt.empty:
    print("   Point settlements — Piping/Overtopping...")
    pt_pip = run_zonal_stats(sett_pt, PIP_R, OVT_R, PIP_ARR_REL, OVT_ARR_REL,
                             HZ_PIP_TIF, HZ_OVT_TIF)
    pt_ovt = run_zonal_stats(sett_pt, PIP_R, OVT_R, PIP_ARR_REL, OVT_ARR_REL,
                             HZ_PIP_TIF, HZ_OVT_TIF)

# Confirm columns
if not sett_ply_smooth.empty:
    zc = [c for c in ply_pip.columns
          if any(c.startswith(p) for p in ["pip_dep","pip_vel","pip_wse","pip_arr",
                                            "ovt_dep","ovt_vel","ovt_wse","ovt_arr"])]
    print(f"\n✅ Zonal stat columns created: {zc}")

🔄 Running zonal statistics...
   Polygon settlements — Piping...
   Polygon settlements — Overtopping...

✅ Zonal stat columns created: ['pip_dep_max', 'pip_dep_avg', 'ovt_dep_max', 'ovt_dep_avg', 'pip_vel_max', 'pip_vel_avg', 'ovt_vel_max', 'ovt_vel_avg', 'pip_wse_max', 'pip_wse_avg', 'ovt_wse_max', 'ovt_wse_avg', 'pip_arr_min', 'pip_arr_avg', 'ovt_arr_min', 'ovt_arr_avg']


## Cell 14 — PAR Calculation + Flood Status Callouts

In [ ]:
# Cell 14
def calc_par(gdf, inund_gdf, sc, pop_col, pop_m, pop_f, hh_col, roads_gdf):
    """
    Adds per settlement:
      {sc}_orig_ha, {sc}_inund_ha, {sc}_inund_pct
      {sc}_PAR, {sc}_PAR_m, {sc}_PAR_f, {sc}_PAR_hh
      {sc}_flood_st, {sc}_surrounded, {sc}_isolated
      inundated (bool)
    """
    inund_u  = inund_gdf.geometry.unary_union
    result   = gdf.copy()
    road_u   = (roads_gdf.geometry.unary_union
                if roads_gdf is not None and not roads_gdf.empty else None)

    def si(val):
        try: return 0 if val is None or np.isnan(float(val)) else int(float(val))
        except: return 0

    o_ha_l, i_ha_l, i_pct_l = [], [], []
    par_l, par_m_l, par_f_l, par_hh_l = [], [], [], []
    fst_l, sur_l, iso_l = [], [], []

    for _, row in result.iterrows():
        geom  = row.geometry
        o_ha  = geom.area / 10000
        inter = geom.intersection(inund_u)
        i_ha  = inter.area / 10000 if not inter.is_empty else 0.0
        i_pct = round(i_ha / o_ha * 100, 2) if o_ha > 0 else 0.0

        sur = inund_u.contains(geom) or i_pct >= 98.0
        iso = False
        if road_u and i_ha > 0:
            buf_500 = geom.buffer(500)
            nr      = road_u.intersection(buf_500)
            if not nr.is_empty and nr.length > 0:
                iso = (nr.intersection(inund_u).length / nr.length * 100) >= 90.0

        pop = si(row.get(pop_col,  0))
        pm  = si(row.get(pop_m,    0)) if pop_m and pop_m  in row.index else 0
        pf  = si(row.get(pop_f,    0)) if pop_f and pop_f  in row.index else 0
        hh  = si(row.get(hh_col,   0)) if hh_col and hh_col in row.index else 0
        r   = i_pct / 100.0

        o_ha_l.append(round(o_ha, 4));  i_ha_l.append(round(i_ha, 4))
        i_pct_l.append(i_pct)
        par_l.append(round(pop * r));   par_m_l.append(round(pm * r))
        par_f_l.append(round(pf * r));  par_hh_l.append(round(hh * r))
        fst_l.append(flood_status_label(i_pct, sur, iso))
        sur_l.append(sur);  iso_l.append(iso)

    result[f"{sc}_orig_ha"]    = o_ha_l
    result[f"{sc}_inund_ha"]   = i_ha_l
    result[f"{sc}_inund_pct"]  = i_pct_l
    result[f"{sc}_PAR"]        = par_l
    result[f"{sc}_PAR_m"]      = par_m_l
    result[f"{sc}_PAR_f"]      = par_f_l
    result[f"{sc}_PAR_hh"]     = par_hh_l
    result[f"{sc}_flood_st"]   = fst_l
    result[f"{sc}_surrounded"] = sur_l
    result[f"{sc}_isolated"]   = iso_l
    result["inundated"]        = [p > 0 for p in i_pct_l]
    # Ensure POP_COLUMN is numeric
    if pop_col in result.columns:
        result[pop_col] = pd.to_numeric(result[pop_col], errors="coerce").fillna(0).astype(int)
    return result


print("🔄 Calculating PAR and flood status...")
if not sett_ply_smooth.empty:
    ply_pip = calc_par(ply_pip, inund_pip, "pip",
                       POP_COLUMN, POP_MALE_COLUMN, POP_FEMALE_COLUMN,
                       HH_COLUMN, osm_roads)
    ply_ovt = calc_par(ply_ovt, inund_ovt, "ovt",
                       POP_COLUMN, POP_MALE_COLUMN, POP_FEMALE_COLUMN,
                       HH_COLUMN, osm_roads)
    for df, sc, lbl in [(ply_pip,"pip","PIPING"), (ply_ovt,"ovt","OVERTOPPING")]:
        aff   = df[df["inundated"]]
        tot_p = int(df[POP_COLUMN].sum())
        tot_r = int(df[f"{sc}_PAR"].sum())
        print(f"\n── {lbl} ─────────────────────────────────────")
        print(f"   Assessed : {len(df)} villages  |  Inundated: {len(aff)}")
        print(f"   Total pop: {tot_p:,}  |  PAR: {tot_r:,}  ({tot_r/tot_p*100:.1f}%)" if tot_p else "   Population data not available")
        for st in ["VILLAGE COMPLETELY FLOODED","VILLAGE SURROUNDED BY FLOOD",
                   "VILLAGE ISOLATED — ROAD ACCESS CUT","VILLAGE HEAVILY FLOODED",
                   "VILLAGE MODERATELY FLOODED","VILLAGE PARTIALLY FLOODED"]:
            cnt = (df[f"{sc}_flood_st"] == st).sum()
            if cnt > 0: print(f"   {st}: {cnt}")

if not sett_pt.empty:
    pt_pip["inundated"] = pt_pip.geometry.within(inund_pip.geometry.unary_union)
    pt_ovt["inundated"] = pt_ovt.geometry.within(inund_ovt.geometry.unary_union)
    for df, sc in [(pt_pip,"pip"), (pt_ovt,"ovt")]:
        df[f"{sc}_PAR"] = df.apply(
            lambda r: si(r.get(POP_COLUMN, 0)) if r["inundated"] else 0, axis=1)

print("\n✅ PAR complete.")

🔄 Calculating PAR and flood status...

── PIPING ─────────────────────────────────────
   Assessed : 22 villages  |  Inundated: 9
   Total pop: 69,829  |  PAR: 1,225  (1.8%)
   VILLAGE PARTIALLY FLOODED: 6

── OVERTOPPING ─────────────────────────────────────
   Assessed : 22 villages  |  Inundated: 10
   Total pop: 69,829  |  PAR: 3,598  (5.2%)
   VILLAGE MODERATELY FLOODED: 4
   VILLAGE PARTIALLY FLOODED: 4

✅ PAR complete.


## Cell 15 — Evacuation Priority + Vulnerability Index

In [ ]:
# Cell 15
def add_evac_vulnerability(gdf, sc):
    """
    Reads pip_arr_min / ovt_arr_min (relative hours after breach).
    Villages near dam get P1 IMMEDIATE — this is CORRECT behaviour.
    Vulnerability index = weighted combination of hazard, inundation %, PAR, distance.
    """
    result  = gdf.copy()
    arr_col = f"{sc}_arr_min"
    arrivals = (result[arr_col].tolist()
                if arr_col in result.columns else [np.nan]*len(result))

    evac = [evac_priority(t) for t in arrivals]
    result[f"{sc}_ev_ord"] = [e[0] for e in evac]
    result[f"{sc}_ev_win"] = [e[1] for e in evac]   # '0 to 2 hour' etc.
    result[f"{sc}_ev_cod"] = [e[2] for e in evac]   # P1–P5
    result[f"{sc}_ev_lbl"] = [e[3] for e in evac]   # IMMEDIATE etc.

    # Vulnerability index (0–1)
    def norm(s):
        mn, mx = s.min(), s.max()
        return (s - mn) / (mx - mn + 1e-9)

    H = norm(pd.to_numeric(result.get(f"{sc}_hz_cod",  pd.Series([0]*len(result))), errors="coerce").fillna(0))
    I = norm(pd.to_numeric(result.get(f"{sc}_inund_pct",pd.Series([0]*len(result))), errors="coerce").fillna(0))
    P = norm(pd.to_numeric(result.get(f"{sc}_PAR",      pd.Series([0]*len(result))), errors="coerce").fillna(0))
    D = 1 - norm(pd.to_numeric(result.get(f"{sc}_dist_km", pd.Series([999]*len(result))), errors="coerce").fillna(999).clip(0))

    result[f"{sc}_v_idx"] = (0.35*H + 0.30*I + 0.20*P + 0.15*D).round(4)
    result[f"{sc}_v_cat"] = result[f"{sc}_v_idx"].apply(
        lambda v: "Critical" if v >= 0.80 else
                  "Very High" if v >= 0.60 else
                  "High"      if v >= 0.40 else
                  "Medium"    if v >= 0.20 else "Low")
    return result


if not sett_ply_smooth.empty:
    ply_pip = add_evac_vulnerability(ply_pip, "pip")
    ply_ovt = add_evac_vulnerability(ply_ovt, "ovt")
if not sett_pt.empty:
    pt_pip  = add_evac_vulnerability(pt_pip,  "pip")
    pt_ovt  = add_evac_vulnerability(pt_ovt,  "ovt")
print("✅ Evacuation priority + vulnerability index assigned.")

✅ Evacuation priority + vulnerability index assigned.


## Cell 16 — Nearest Shelter Assignment

In [ ]:
# Cell 16
def nearest_shelter(gdf, sh_gdf, name_col, cap_col):
    result = gdf.copy()
    if sh_gdf is None or sh_gdf.empty:
        result["sh_name"]    = "No shelter data"
        result["sh_dist_km"] = np.nan
        return result
    sh_pts = (sh_gdf.geometry.centroid
              if sh_gdf.geometry.geom_type.iloc[0] != "Point"
              else sh_gdf.geometry)
    names, dists, caps = [], [], []
    for geom in gdf.geometry:
        pt  = geom.centroid if geom.geom_type != "Point" else geom
        idx = sh_pts.distance(pt).idxmin()
        row = sh_gdf.loc[idx]
        names.append(str(row.get(name_col, "Unknown")) if name_col in row.index else "Unknown")
        dists.append(round(pt.distance(sh_pts.loc[idx]) / 1000, 3))
        caps.append(row.get(cap_col, None) if cap_col and cap_col in row.index else None)
    result["sh_name"]    = names
    result["sh_dist_km"] = dists
    if any(c is not None for c in caps):
        result["sh_cap"] = caps
    return result


if not sett_ply_smooth.empty:
    ply_pip = nearest_shelter(ply_pip, shelters, SHELTER_NAME_COL, SHELTER_CAP_COL)
    ply_ovt = nearest_shelter(ply_ovt, shelters, SHELTER_NAME_COL, SHELTER_CAP_COL)
if not sett_pt.empty:
    pt_pip  = nearest_shelter(pt_pip,  shelters, SHELTER_NAME_COL, SHELTER_CAP_COL)
    pt_ovt  = nearest_shelter(pt_ovt,  shelters, SHELTER_NAME_COL, SHELTER_CAP_COL)
print("✅ Nearest shelter assigned.")

✅ Nearest shelter assigned.


## Cell 17 — Road & POI Inundation Analysis

In [ ]:
# ── CELL 17A — Road geometry prep + distance from dam ───────────────
# (no rasters touched here — pure vector, very low RAM)

import gc
import numpy as np
from rasterstats import zonal_stats

if osm_roads is None or osm_roads.empty:
    print("⚠️  No OSM roads — skipping road analysis.")
    roads_final = gpd.GeoDataFrame()
else:
    roads_final = osm_roads.copy()

    # Total length
    roads_final["total_km"] = (roads_final.geometry.length / 1000).round(4)

    # Distance from dam along river (midpoint of each road segment)
    riv  = river.geometry.unary_union
    line = (max(riv.geoms, key=lambda l: l.length)
            if hasattr(riv, "geoms") else riv)
    d0   = line.project(dam_point)

    roads_final["dist_dam_km"] = [
        round(abs(line.project(g.interpolate(0.5, normalized=True)) - d0) / 1000, 3)
        for g in roads_final.geometry
    ]

    # Free river union — not needed anymore
    del riv, line, d0
    gc.collect()

    print(f"✅ 17A done — {len(roads_final)} road segments prepped")
    print(f"   total_km range: {roads_final['total_km'].min():.3f} – "
          f"{roads_final['total_km'].max():.3f} km")


✅ 17A done — 2653 road segments prepped
   total_km range: 0.003 – 513.631 km


In [ ]:
# ── CELL 17B — Road inundation geometry (intersection loop) ─────────
# Most RAM-intensive step. Processes one scenario at a time,
# deletes the inundation union immediately after.

if not roads_final.empty:
    for sc, inund_gdf in [("pip", inund_pip), ("ovt", inund_ovt)]:
        print(f"🔄 [{sc}] Computing road inundation extents...")

        # Build union once, use it, delete it
        inund_u = inund_gdf.geometry.unary_union

        ik_l, ip_l, st_l = [], [], []
        for geom, tkm in zip(roads_final.geometry, roads_final["total_km"]):
            clip = geom.intersection(inund_u)
            ikm  = round(clip.length / 1000, 4) if not clip.is_empty else 0.0
            ipct = round(ikm / tkm * 100, 1) if tkm > 0 else 0.0
            if   ipct >= 95: st = "ROAD COMPLETELY FLOODED"
            elif ipct >= 70: st = "ROAD HEAVILY FLOODED"
            elif ipct >= 30: st = "ROAD PARTIALLY FLOODED"
            elif ipct >  0:  st = "ROAD MARGINALLY AFFECTED"
            else:            st = "NOT FLOODED"
            ik_l.append(ikm); ip_l.append(ipct); st_l.append(st)

        roads_final[f"{sc}_inund_km"]  = ik_l
        roads_final[f"{sc}_inund_pct"] = ip_l
        roads_final[f"{sc}_flood_st"]  = st_l
        roads_final[f"{sc}_inundated"] = roads_final[f"{sc}_inund_km"] > 0

        # Free immediately
        del inund_u, ik_l, ip_l, st_l
        gc.collect()

        flooded = int(roads_final[f"{sc}_inundated"].sum())
        km      = roads_final[f"{sc}_inund_km"].sum()
        print(f"   ✅ [{sc}] {flooded} segments flooded | {km:.2f} km")

    print("\n✅ 17B done — road inundation extents computed")



🔄 [pip] Computing road inundation extents...
   ✅ [pip] 138 segments flooded | 32.56 km
🔄 [ovt] Computing road inundation extents...
   ✅ [ovt] 206 segments flooded | 54.59 km

✅ 17B done — road inundation extents computed


In [ ]:
# ══════════════════════════════════════════════════════════════
#  ROAD LOCATION NAME FIX
#  Paste as a NEW CELL immediately after Cell 17B.
#  Replaces "Unnamed Village Road (OSM:xxxxx)" with
#  location-based names like:
#    "Village Road — Motyal"
#    "SH 151 — Motyal to Sangavi"
#    "Kutcha Track — Near Kolekarwadi"
# ══════════════════════════════════════════════════════════════

import gc

if not roads_final.empty:

    # ── Step 1: Spatial join roads → settlements to get village name ────
    # Use the smoothed settlement polygons (already in memory)
    # Join each road's midpoint to whichever village polygon contains it

    # Build a points GDF from road midpoints — lightweight
    import geopandas as gpd
    from shapely.geometry import Point

    mid_pts = gpd.GeoDataFrame(
        {"road_idx": roads_final.index},
        geometry=[g.interpolate(0.5, normalized=True) for g in roads_final.geometry],
        crs=TARGET_CRS
    )

    # Use the settlement layer for the join
    # Pick whichever is available: smoothed polygon > raw polygon > point
    if 'sett_ply_smooth' in dir() and not sett_ply_smooth.empty:
        settle_ref = sett_ply_smooth[[NAME_COLUMN, 'geometry']].copy()
    elif 'sett_ply' in dir() and not sett_ply.empty:
        settle_ref = sett_ply[[NAME_COLUMN, 'geometry']].copy()
    else:
        settle_ref = None

    # Spatial join: midpoint within settlement polygon
    if settle_ref is not None:
        joined = gpd.sjoin(
            mid_pts, settle_ref,
            how="left", predicate="within"
        )
        # Map road_idx → village name
        village_map = joined.set_index("road_idx")[NAME_COLUMN].to_dict()
    else:
        village_map = {}

    del mid_pts
    gc.collect()

    # ── Step 2: Nearest settlement fallback (for roads outside polygons) ─
    # For roads that didn't fall inside any polygon, find closest centroid

    if settle_ref is not None:
        settle_pts = settle_ref.copy()
        settle_pts["geometry"] = settle_pts.geometry.centroid

        # Only compute for roads that got no village from the join
        unmapped_idx = [i for i in roads_final.index if village_map.get(i) is None
                         or str(village_map.get(i)) in ('', 'nan', 'None')]

        if unmapped_idx:
            unmapped_geoms = roads_final.loc[unmapped_idx, "geometry"].apply(
                lambda g: g.interpolate(0.5, normalized=True))

            for idx, midpt in zip(unmapped_idx, unmapped_geoms):
                dists = settle_pts.geometry.distance(midpt)
                nearest_idx = dists.idxmin()
                nearest_km  = dists.min() / 1000
                vname = settle_pts.loc[nearest_idx, NAME_COLUMN]
                # Only assign if within 3 km — beyond that it's too far to be meaningful
                village_map[idx] = vname if nearest_km <= 3.0 else None

        del settle_pts, settle_ref
        gc.collect()

    # ── Step 3: Build clean EAP-ready road name ───────────────────────────

    def eap_road_name(row, village_map):
        """
        Priority:
          1. ref (NH/SH/MDR) + village → "SH 151 — Motyal"
          2. ref only                  → "SH 151"
          3. real OSM name + village   → "Akkalkot Road — Motyal"
          4. real OSM name only        → "Akkalkot Road"
          5. road category + village   → "Village Road — Motyal"
          6. road category only        → "Village Road"
        Never shows OSM IDs.
        """
        idx      = row.name
        ref      = str(row.get("ref", "") or "").strip()
        raw_name = str(row.get("name", "") or "").strip()
        cat      = str(row.get("road_cat", "Road")).strip()
        village  = village_map.get(idx)
        vstr     = f" — {village}" if village and str(village) not in ('nan','None','') else ""

        # Strip out any previously generated "Unnamed ... (OSM:...)" labels
        if raw_name.startswith("Unnamed ") or "OSM:" in raw_name:
            raw_name = ""

        if ref and raw_name:
            return f"{ref} — {raw_name}{vstr}"
        elif ref:
            return f"{ref}{vstr}"
        elif raw_name:
            return f"{raw_name}{vstr}"
        else:
            return f"{cat}{vstr}"

    roads_final["name"] = roads_final.apply(
        lambda row: eap_road_name(row, village_map), axis=1)

    del village_map
    gc.collect()

    # ── Step 4: Preview ───────────────────────────────────────────────────
    print("✅ Road names updated with location context")
    print("\nSample named roads (flooded only):")
    flooded_preview = roads_final[
        roads_final["pip_inundated"] | roads_final["ovt_inundated"]
    ][["name", "road_cat", "pip_inund_km", "ovt_inund_km", "dist_dam_km"]].head(15)

    from IPython.display import display
    display(flooded_preview)

✅ Road names updated with location context

Sample named roads (flooded only):


,name,road_cat,pip_inund_km,ovt_inund_km,dist_dam_km
25,SH74 — SH74 — Dahithane,Major Road,0.9333,3.1254,18.135
26,SH74 — SH74 — Waluj,Major Road,0.1074,0.1124,23.824
27,SH74 — SH74 — Waluj,Major Road,0.0000,0.1871,24.718
36,Village Road — Hattij,Village Road,0.0645,0.4178,3.819
38,MDR28 — MDR28 — Sasure,District Road,0.6001,1.2770,13.970
39,SH151 — SH151 — Raleras,Major Road,0.1061,0.1061,11.790
40,SH151 — SH151 — Raleras,Major Road,0.0692,0.4866,12.597
113,Village Road — Hattij,Village Road,0.0840,0.0840,2.902
114,Village Road — Hattij,Village Road,0.2153,0.2861,3.068
117,Internal Road — Raleras,Internal Road,0.0000,0.1837,11.899


In [ ]:
# ══════════════════════════════════════════════════════════════
#  FACILITY / AMENITY LOCATION NAME FIX
#  Paste as a NEW CELL immediately after the road location fix.
#  Replaces "Unnamed Bridge (OSM:297985343)" with
#  location-based EAP names like:
#    "Bridge — Motyal (on Village Road)"
#    "Primary Health Centre — Sangavi"
#    "School — Near Kolekarwadi"
#    "Water Well — Kalegaon"
# ══════════════════════════════════════════════════════════════

import gc
import geopandas as gpd
import numpy as np
from IPython.display import display

if poi_final is not None and not poi_final.empty:

    # ── Step 1: Spatial join POI points → settlement polygons ───────────
    if 'sett_ply_smooth' in dir() and not sett_ply_smooth.empty:
        settle_ref = sett_ply_smooth[[NAME_COLUMN, 'geometry']].copy()
    elif 'sett_ply' in dir() and not sett_ply.empty:
        settle_ref = sett_ply[[NAME_COLUMN, 'geometry']].copy()
    else:
        settle_ref = None

    if settle_ref is not None:
        # Direct point-in-polygon join
        poi_join = gpd.sjoin(
            poi_final[['geometry']].copy(),
            settle_ref,
            how="left",
            predicate="within"
        )
        village_map = poi_join[NAME_COLUMN].to_dict()

        # Nearest-settlement fallback for POI outside any polygon
        settle_pts = settle_ref.copy()
        settle_pts["geometry"] = settle_pts.geometry.centroid

        unmapped = [i for i in poi_final.index
                    if village_map.get(i) is None
                    or str(village_map.get(i)) in ('', 'nan', 'None')]

        for idx in unmapped:
            pt   = poi_final.loc[idx, "geometry"]
            dist = settle_pts.geometry.distance(pt)
            nidx = dist.idxmin()
            nkm  = dist.min() / 1000
            village_map[idx] = settle_pts.loc[nidx, NAME_COLUMN] if nkm <= 5.0 else None

        del settle_pts, settle_ref, poi_join
        gc.collect()
    else:
        village_map = {}

    # ── Step 2: EAP category labels (plain English, no OSM jargon) ───────
    # Maps internal category strings → EAP report terminology
    EAP_CATEGORY = {
        "Grampanchayat/Municipal":  "Grampanchayat / Municipal Office",
        "Hospital":                 "Hospital",
        "PHC/Clinic":               "Primary Health Centre (PHC)",
        "Pharmacy":                 "Pharmacy",
        "School":                   "School",
        "College":                  "College / University",
        "Anganwadi/Kindergarten":   "Anganwadi Centre",
        "Anganwadi/Social Centre":  "Anganwadi / Social Centre",
        "Temple/Religious":         "Temple / Place of Worship",
        "Police Station":           "Police Station",
        "Fire Station":             "Fire Station",
        "Bank":                     "Bank",
        "ATM":                      "ATM",
        "Post Office":              "Post Office",
        "Bus Stop":                 "Bus Stop",
        "Bus Station":              "Bus Station",
        "Market/Bazaar":            "Market / Bazaar",
        "Water Well":               "Water Well",
        "Water Tower/Tank":         "Water Tower / Storage Tank",
        "Power Substation":         "Electricity Substation",
        "Community Centre":         "Community / Village Hall",
        "Shelter":                  "Emergency Shelter",
        "Bridge":                   "Bridge",
        "Canal":                    "Canal",
        "Weir":                     "Weir",
        "Barrage":                  "Barrage",
        "Dam":                      "Dam",
        "Other":                    "Other Facility",
    }

    # ── Step 3: Build clean EAP facility name ────────────────────────────
    def eap_facility_name(row, village_map, eap_cat):
        """
        Priority:
          1. Real OSM name + village  → "Sai Hospital — Motyal"
          2. Real OSM name only       → "Sai Hospital"
          3. Category label + village → "Primary Health Centre — Sangavi"
          4. Category label only      → "Primary Health Centre"
          Never shows OSM IDs.
        """
        idx     = row.name
        raw     = str(row.get("name", "") or "").strip()
        cat     = str(row.get("category", "Other") or "Other")
        cat_lbl = eap_cat.get(cat, cat)
        village = village_map.get(idx)
        vstr    = (f" — {village}"
                   if village and str(village) not in ('nan', 'None', '') else "")

        # Remove previously generated "Unnamed ... (OSM:...)" labels
        if (not raw
                or raw.startswith("Unnamed ")
                or "OSM:" in raw
                or raw == cat_lbl):
            # No usable name → use category + location
            return f"{cat_lbl}{vstr}"
        else:
            return f"{raw}{vstr}"

    poi_final["name"]     = poi_final.apply(
        lambda r: eap_facility_name(r, village_map, EAP_CATEGORY), axis=1)

    # Also update the category column to EAP terminology
    poi_final["category"] = poi_final["category"].map(
        lambda c: EAP_CATEGORY.get(str(c), str(c)))

    del village_map
    gc.collect()

    # ── Step 4: Drop Water Wells from the DISPLAY table ──────────────────
    # Water wells bulk out the table with 200+ unnamed rows
    # They are saved to CSV/SHP for GIS use but hidden from the Colab table
    EXCLUDE_FROM_DISPLAY = {"Water Well", "Water Tower / Storage Tank"}

    poi_display = poi_final[
        ~poi_final["category"].isin(EXCLUDE_FROM_DISPLAY)
    ].copy()

    print(f"✅ Facility names updated with location context")
    print(f"   Total POI    : {len(poi_final)}")
    print(f"   Display POI  : {len(poi_display)}  "
          f"(Water Wells saved to CSV but hidden from table)")

    print("\n📊 Category breakdown after fix:")
    print(poi_final["category"].value_counts().to_string())

    # ── Step 5: Rebuild the styled display table ──────────────────────────
    # Same colour scheme as before, now with clean names

    def ci(v):  # inundated bool
        return ("background:#c0392b;color:white;font-weight:bold"
                if v is True else "background:#d5f5e3;color:black")

    def cd(v):  # depth
        try:
            f = float(v)
            if f >= 4:   return "background:#7b0000;color:white"
            if f >= 2:   return "background:#c0392b;color:white"
            if f >= 1:   return "background:#e67e22;color:black"
            if f >= 0.5: return "background:#f9e79f;color:black"
            if f >  0:   return "background:#d5f5e3;color:black"
        except: pass
        return ""

    def ce(v):  # evac code
        return {
            "P1": "background:#c0392b;color:white;font-weight:bold",
            "P2": "background:#e67e22;color:white",
            "P3": "background:#f39c12;color:black",
            "P4": "background:#82e0aa;color:black",
            "P5": "background:#d5f5e3;color:black",
        }.get(str(v), "")

    TH = [
        {"selector": "th", "props": [("background","#1a2a3a"), ("color","white"),
                                      ("font-size","12px"), ("padding","6px 10px"),
                                      ("white-space","nowrap")]},
        {"selector": "td", "props": [("font-size","11px"), ("padding","4px 8px"),
                                      ("border-bottom","1px solid #ddd")]},
        {"selector": "tr:hover td", "props": [("background","#f0f7ff")]},
    ]

    # ── TABLE A: Facilities by priority (flooded first, then by distance) ─
    display(display.__class__.__mro__)  # dummy to confirm display imported
    from IPython.display import display as ipy_display, HTML

    # Sort: flooded first, then by distance
    poi_show = poi_display.copy()
    poi_show["_flooded_any"] = (
        poi_show.get("pip_inund", False) | poi_show.get("ovt_inund", False))
    poi_show = poi_show.sort_values(
        ["_flooded_any", "dist_dam_km"], ascending=[False, True])
    poi_show = poi_show.drop(columns=["_flooded_any"])

    # Columns for EAP report
    EAP_COLS = {
        "category":    "Facility Type",
        "name":        "Facility Name",
        "dist_dam_km": "Dist. from Dam (km)",
        "pip_inund":   "Piping — Flooded?",
        "pip_dep_max": "Piping — Max Depth (m)",
        "pip_arr_min": "Piping — Arrival Time (hrs)",
        "pip_ev_win":  "Piping — Evac Window",
        "ovt_inund":   "Overtop — Flooded?",
        "ovt_dep_max": "Overtop — Max Depth (m)",
        "ovt_arr_min": "Overtop — Arrival Time (hrs)",
        "ovt_ev_win":  "Overtop — Evac Window",
    }
    use_cols = [c for c in EAP_COLS if c in poi_show.columns]
    df_tbl   = poi_show[use_cols].rename(columns=EAP_COLS).copy()

    # Replace NaN in object cols with dash
    for c in df_tbl.select_dtypes("object").columns:
        df_tbl[c] = df_tbl[c].replace({"": "—", "nan": "—", "None": "—"}).fillna("—")

    inund_c = [v for k, v in EAP_COLS.items() if "inund" in k   and v in df_tbl.columns]
    dep_c   = [v for k, v in EAP_COLS.items() if "dep"   in k   and v in df_tbl.columns]
    ev_c    = [v for k, v in EAP_COLS.items() if "ev_cod" in k  and v in df_tbl.columns]

    ipy_display(HTML(
        "<h3 style='color:#4a235a'>"
        "🏛️ Public Facilities — Flood Status & Distance from Jawalgaon Dam"
        "</h3>"
        "<p style='font-size:12px;color:#555'>"
        "Sorted: flooded facilities first, then by distance from dam. "
        "Water wells excluded from this table (saved to CSV separately)."
        "</p>"
    ))

    styled = (df_tbl.style
              .applymap(ci, subset=inund_c)
              .applymap(cd, subset=dep_c)
              .format({c: "{:.3f}" for c in df_tbl.select_dtypes("float").columns})
              .set_table_styles(TH)
              .set_caption(
                  f"Public Facilities — {len(df_tbl)} features | "
                  f"{DAM_NAME} Dam Break Impact Assessment"))
    ipy_display(styled)

    # ── TABLE B: Category summary ─────────────────────────────────────────
    ipy_display(HTML(
        "<h3 style='color:#4a235a'>📊 Facility Category Summary</h3>"))

    pip_ic = "pip_inund" if "pip_inund" in poi_final.columns else None
    ovt_ic = "ovt_inund" if "ovt_inund" in poi_final.columns else None
    pip_dm = "pip_dep_max" if "pip_dep_max" in poi_final.columns else None
    ovt_dm = "ovt_dep_max" if "ovt_dep_max" in poi_final.columns else None
    pip_am = "pip_arr_min" if "pip_arr_min" in poi_final.columns else None

    agg = {"Total": ("category", "count")}
    if pip_ic: agg["Pip Flooded"]    = (pip_ic, "sum")
    if ovt_ic: agg["Ovt Flooded"]    = (ovt_ic, "sum")
    if pip_dm: agg["Pip Max Depth"]  = (pip_dm, "max")
    if ovt_dm: agg["Ovt Max Depth"]  = (ovt_dm, "max")
    if pip_am: agg["Pip Min Arrival"]= (pip_am, "min")
    if "dist_dam_km" in poi_final.columns:
        agg["Nearest (km)"] = ("dist_dam_km", "min")

    summ = (poi_final.groupby("category")
            .agg(**{k: v for k, v in agg.items()})
            .reset_index()
            .rename(columns={"category": "Facility Type"}))

    if "Nearest (km)" in summ.columns:
        summ = summ.sort_values("Nearest (km)")

    grad_c = [c for c in summ.columns if "Flooded" in c or "Depth" in c]
    ipy_display(
        summ.style
        .background_gradient(subset=grad_c, cmap="Reds")
        .format({c: "{:.2f}" for c in summ.select_dtypes("float").columns})
        .set_table_styles(TH)
        .set_caption("Facility Category Summary — Flooded count, Max Depth, Earliest Arrival")
    )

    # ── Step 6: Save updated CSVs ─────────────────────────────────────────
    poi_final.drop(columns="geometry").to_csv(
        os.path.join(OUT_DIR, "OSM_POI_Flood_Status.csv"), index=False)
    # Separate water wells CSV
    wells = poi_final[poi_final["category"].isin(
        {"Water Well", "Water Tower / Storage Tank"})]
    if not wells.empty:
        wells.drop(columns="geometry").to_csv(
            os.path.join(OUT_DIR, "OSM_Water_Wells.csv"), index=False)
    print(f"\n✅ CSVs saved to {OUT_DIR}")

else:
    print("⚠️  poi_final is empty — run Cell 17D first.")

✅ Facility names updated with location context
   Total POI    : 361
   Display POI  : 175  (Water Wells saved to CSV but hidden from table)

📊 Category breakdown after fix:
category
Water Well                     186
Bridge                         147
Hospital                         9
Bus Stop                         4
Temple / Place of Worship        4
Primary Health Centre (PHC)      3
Electricity Substation           3
School                           2
Dam                              1
Bus Station                      1
College / University             1


(function, object)

,Facility Type,Facility Name,Dist. from Dam (km),Piping — Flooded?,Piping — Max Depth (m),Piping — Arrival Time (hrs),Piping — Evac Window,Overtop — Flooded?,Overtop — Max Depth (m),Overtop — Arrival Time (hrs),Overtop — Evac Window
119,Bridge,Bridge — Jotibachiwadi — Jotibachiwadi — Jotibachiwadi,0.081,True,6.285,2.658,2–4 hours,True,8.250,0.656,0 to 2 hour
121,Bridge,Bridge — Jotibachiwadi — Jotibachiwadi — Jotibachiwadi,0.999,True,6.315,2.803,2–4 hours,True,7.614,0.740,0 to 2 hour
127,Bridge,Bridge — Jotibachiwadi — Jotibachiwadi — Jotibachiwadi,1.156,True,6.149,2.811,2–4 hours,True,7.384,0.759,0 to 2 hour
126,Bridge,Bridge — Hattij — Hattij — Hattij,1.638,True,5.856,2.859,2–4 hours,True,7.164,0.961,0 to 2 hour
120,Bridge,Bridge — Hattij — Hattij — Hattij,2.075,True,5.310,3.355,2–4 hours,True,7.381,1.113,0 to 2 hour
151,Bridge,Bridge — Hattij — Hattij — Hattij,2.260,True,2.797,3.484,2–4 hours,True,5.026,1.165,0 to 2 hour
115,Bridge,Bridge — Hattij — Hattij — Hattij,2.902,True,6.000,3.574,2–4 hours,True,8.067,1.208,0 to 2 hour
129,Bridge,Bridge — Hattij — Hattij — Hattij,3.578,True,5.888,3.761,2–4 hours,True,8.330,1.255,0 to 2 hour
130,Bridge,Bridge — Hingani(ratanjan) — Hingani(ratanjan) — Hingani(ratanjan),4.279,True,6.677,3.861,2–4 hours,True,8.717,1.317,0 to 2 hour
164,Bridge,Bridge — Hingani(ratanjan) — Hingani(ratanjan) — Hingani(ratanjan),5.627,True,7.052,4.175,4–6 hours,True,8.975,1.412,0 to 2 hour


,Facility Type,Total,Pip Flooded,Ovt Flooded,Pip Max Depth,Ovt Max Depth,Pip Min Arrival,Nearest (km)
0,Bridge,147,19,21,7.05,8.97,2.66,0.00
1,Bus Station,1,0,0,nan,nan,nan,0.00
2,Bus Stop,4,0,0,nan,nan,nan,0.00
3,College / University,1,0,0,nan,nan,nan,0.00
4,Dam,1,0,0,nan,nan,nan,0.00
5,Electricity Substation,3,0,0,nan,nan,nan,0.00
6,Hospital,9,0,0,nan,nan,nan,0.00
7,Primary Health Centre (PHC),3,0,0,nan,nan,nan,0.00
8,School,2,0,0,nan,nan,nan,0.00
9,Temple / Place of Worship,4,0,0,nan,nan,nan,0.00



✅ CSVs saved to /content/PAR_Outputs


## Cell 18 — Save All Shapefiles

In [ ]:
# Cell 18
def safe_save(gdf, path):
    """Save GeoDataFrame — truncates column names to 10 chars (shapefile limit)."""
    if gdf is None or (hasattr(gdf, "empty") and gdf.empty):
        print(f"   ⚠️  Skipped (empty): {os.path.basename(path)}")
        return
    out = gdf.copy()
    seen, cols = {}, []
    for c in out.columns:
        t = c[:10]
        if t in seen:
            seen[t] += 1
            cols.append(f"{t[:8]}_{seen[t]}")
        else:
            seen[t] = 0
            cols.append(t)
    out.columns = cols
    out.to_file(path)
    print(f"   ✅ {os.path.basename(path)}")

print("💾 Saving shapefiles...")
safe_save(inund_pip,   os.path.join(OUT_DIR, "inundation_piping.shp"))
safe_save(inund_ovt,   os.path.join(OUT_DIR, "inundation_overtopping.shp"))
safe_save(dam_gdf,     os.path.join(OUT_DIR, "dam_location.shp"))

if not sett_ply_smooth.empty:
    safe_save(ply_pip, os.path.join(OUT_DIR, "settlements_piping.shp"))
    safe_save(ply_ovt, os.path.join(OUT_DIR, "settlements_overtopping.shp"))
if not sett_pt.empty:
    safe_save(pt_pip,  os.path.join(OUT_DIR, "settlements_pt_piping.shp"))
    safe_save(pt_ovt,  os.path.join(OUT_DIR, "settlements_pt_overtopping.shp"))

safe_save(roads_final, os.path.join(OUT_DIR, "roads_inundation.shp"))
safe_save(poi_final,   os.path.join(OUT_DIR, "osm_poi_inundation.shp"))
print("✅ All shapefiles saved.")

💾 Saving shapefiles...
   ✅ inundation_piping.shp
   ✅ inundation_overtopping.shp
   ✅ dam_location.shp
   ✅ settlements_piping.shp
   ✅ settlements_overtopping.shp
   ✅ roads_inundation.shp
   ✅ osm_poi_inundation.shp
✅ All shapefiles saved.


## Cell 19 — Build Annexure 3A & 3B (EAP Standard Format)

In [ ]:
# Cell 19
def build_3A(gdf, sc, scenario_label):
    aff  = gdf[gdf["inundated"]].copy().sort_values(f"{sc}_dist_km")
    rows = []
    for i, (_, r) in enumerate(aff.iterrows(), 1):
        arr  = r.get(f"{sc}_arr_min", np.nan)
        safe = lambda col, dec=3: round(float(r.get(col,0) or 0), dec)
        ev   = evac_priority(arr)
        rows.append({
            "Sl No":                             i,
            "Evacuation Window":                 ev[1],
            "Priority Order":                    ev[0],
            "Priority Code":                     ev[2],
            "Priority Label":                    ev[3],
            "Village Name":                      r.get(NAME_COLUMN, ""),
            "District":                          DAM_DISTRICT,
            f"Dist from {DAM_NAME} (km)":        safe(f"{sc}_dist_km"),
            "Max Depth (m)":                     safe(f"{sc}_dep_max"),
            "Avg Depth (m)":                     safe(f"{sc}_dep_avg"),
            "Max Velocity (m/s)":                safe(f"{sc}_vel_max"),
            "Avg Velocity (m/s)":                safe(f"{sc}_vel_avg"),
            "Max WSE (m)":                       safe(f"{sc}_wse_max"),
            "Flood Arrival Time (hrs)":          round(float(arr),2) if not np.isnan(arr) else "Not Reached",
            "Hazard Class":                      r.get(f"{sc}_hz_cls", "N/A"),
            "Hazard Level":                      r.get(f"{sc}_hz_lbl", "N/A"),
            "Flood Status":                      r.get(f"{sc}_flood_st", "N/A"),
            "Village Surrounded":                "YES" if r.get(f"{sc}_surrounded") else "NO",
            "Road Access Cut":                   "YES" if r.get(f"{sc}_isolated")   else "NO",
            "Nearest Shelter":                   r.get("sh_name",    "Not Assigned"),
            "Shelter Dist (km)":                 safe("sh_dist_km"),
            "Vulnerability Index":               safe(f"{sc}_v_idx", 4),
            "Vulnerability Category":            r.get(f"{sc}_v_cat", "N/A"),
            "Scenario":                          scenario_label,
        })
    return pd.DataFrame(rows)

def build_3B(gdf, sc, scenario_label):
    aff  = gdf[gdf["inundated"]].copy().sort_values(f"{sc}_dist_km")
    rows = []
    safe = lambda row, col, dec=3: round(float(row.get(col,0) or 0), dec)
    for i, (_, r) in enumerate(aff.iterrows(), 1):
        pop  = int(r.get(POP_COLUMN, 0) or 0)
        par  = int(r.get(f"{sc}_PAR",   0) or 0)
        ipct = float(r.get(f"{sc}_inund_pct", 0) or 0)
        arr  = r.get(f"{sc}_arr_min", np.nan)
        ev   = evac_priority(arr)
        rows.append({
            "Sl No":                     i,
            "Village Name":              r.get(NAME_COLUMN, ""),
            "District":                  DAM_DISTRICT,
            "Area Inundated (%)":        f"{ipct:.1f}%",
            "Original Area (ha)":        safe(r, f"{sc}_orig_ha", 2),
            "Inundated Area (ha)":       safe(r, f"{sc}_inund_ha", 2),
            "Existing Population":       pop,
            "Effected Population (PAR)": par,
            "PAR % of Village":          f"{par/pop*100:.1f}%" if pop>0 else "N/A",
            "Max Depth (m)":             safe(r, f"{sc}_dep_max"),
            "Avg Depth (m)":             safe(r, f"{sc}_dep_avg"),
            "Max Velocity (m/s)":        safe(r, f"{sc}_vel_max"),
            "Max WSE (m)":               safe(r, f"{sc}_wse_max"),
            "Flood Arrival (hrs)":       round(float(arr),2) if not np.isnan(arr) else "Not Reached",
            "Hazard Class":              r.get(f"{sc}_hz_cls", "N/A"),
            "Hazard Level":              r.get(f"{sc}_hz_lbl", "N/A"),
            "Evacuation Window":         ev[1],
            "Evacuation Priority":       ev[3],
            "Flood Status Callout":      r.get(f"{sc}_flood_st", "N/A"),
            f"Dist from {DAM_NAME} (km)": safe(r, f"{sc}_dist_km"),
            "Nearest Shelter":           r.get("sh_name", "Not Assigned"),
            "Shelter Dist (km)":         safe(r, "sh_dist_km"),
            "Vulnerability Index":       safe(r, f"{sc}_v_idx", 4),
            "Vulnerability Category":    r.get(f"{sc}_v_cat", "N/A"),
            "Scenario":                  scenario_label,
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        tot_p = df["Existing Population"].sum()
        tot_r = df["Effected Population (PAR)"].sum()
        df = pd.concat([df, pd.DataFrame([{
            "Sl No": "TOTAL",
            "Village Name": f"ALL {len(df)} AFFECTED VILLAGES",
            "Existing Population": tot_p,
            "Effected Population (PAR)": tot_r,
            "PAR % of Village": f"{tot_r/tot_p*100:.1f}%" if tot_p > 0 else "N/A",
            "Scenario": scenario_label,
        }])], ignore_index=True)
    return df

if not sett_ply_smooth.empty:
    ann3A_pip = build_3A(ply_pip, "pip", "Piping")
    ann3B_pip = build_3B(ply_pip, "pip", "Piping")
    ann3A_ovt = build_3A(ply_ovt, "ovt", "Overtopping")
    ann3B_ovt = build_3B(ply_ovt, "ovt", "Overtopping")
    for nm, df in [("Annexure_3A_Piping.csv",      ann3A_pip),
                   ("Annexure_3B_Piping_PAR.csv",   ann3B_pip),
                   ("Annexure_3A_Overtopping.csv",  ann3A_ovt),
                   ("Annexure_3B_Overtopping_PAR.csv", ann3B_ovt)]:
        df.to_csv(os.path.join(OUT_DIR, nm), index=False)
        print(f"   ✅ {nm}")
    print("✅ Annexure 3A & 3B saved.")

   ✅ Annexure_3A_Piping.csv
   ✅ Annexure_3B_Piping_PAR.csv
   ✅ Annexure_3A_Overtopping.csv
   ✅ Annexure_3B_Overtopping_PAR.csv
✅ Annexure 3A & 3B saved.


## Cell 20 — Full Colour-coded Visualizations

In [ ]:
# Cell 20 — style helpers + all tables
TH = [
    {"selector":"th",         "props":[("background","#1a2a3a"),("color","white"),
                                        ("font-size","12px"),("padding","6px 10px"),
                                        ("white-space","nowrap")]},
    {"selector":"td",         "props":[("font-size","11px"),("padding","4px 8px"),
                                        ("border-bottom","1px solid #ddd")]},
    {"selector":"tr:hover td","props":[("background","#f0f7ff")]},
    {"selector":"caption",    "props":[("caption-side","top"),("font-size","14px"),
                                        ("font-weight","bold"),("padding","8px 0"),
                                        ("color","#1a2a3a")]},
]

def cs(v):  # flood status
    return {"VILLAGE COMPLETELY FLOODED":"background:#7b0000;color:white;font-weight:bold",
            "VILLAGE SURROUNDED BY FLOOD":"background:#c0392b;color:white;font-weight:bold",
            "VILLAGE ISOLATED — ROAD ACCESS CUT":"background:#e74c3c;color:white",
            "VILLAGE HEAVILY FLOODED":"background:#e67e22;color:white",
            "VILLAGE MODERATELY FLOODED":"background:#f39c12;color:black",
            "VILLAGE PARTIALLY FLOODED":"background:#f9e79f;color:black",
            "VILLAGE MARGINALLY AFFECTED":"background:#d5f5e3;color:black",
    }.get(str(v), "")

def ch(v):  # hazard
    return {"H6":"background:#7b0000;color:white;font-weight:bold",
            "H5":"background:#c0392b;color:white",
            "H4":"background:#e67e22;color:black",
            "H3":"background:#f39c12;color:black",
            "H2":"background:#82e0aa;color:black",
            "H1":"background:#1e8449;color:white",
    }.get(str(v), "")

def ce(v):  # evac code
    return {"P1":"background:#c0392b;color:white;font-weight:bold",
            "P2":"background:#e67e22;color:white",
            "P3":"background:#f39c12;color:black",
            "P4":"background:#82e0aa;color:black",
            "P5":"background:#d5f5e3;color:black",
    }.get(str(v), "")

def cel(v):  # evac label
    return {"IMMEDIATE":"background:#c0392b;color:white;font-weight:bold",
            "URGENT":   "background:#e67e22;color:white",
            "HIGH":     "background:#f39c12;color:black",
            "MODERATE": "background:#82e0aa;color:black",
            "PLANNED":  "background:#d5f5e3;color:black",
    }.get(str(v), "")

def cd(v):  # depth
    try:
        f = float(v)
        if f >= 4:   return "background:#7b0000;color:white"
        if f >= 2:   return "background:#c0392b;color:white"
        if f >= 1:   return "background:#e67e22;color:black"
        if f >= 0.5: return "background:#f9e79f;color:black"
        if f >  0:   return "background:#d5f5e3;color:black"
    except: pass
    return ""

def ci(v):  # inundated bool
    return "background:#c0392b;color:white;font-weight:bold" if v is True else "background:#d5f5e3;color:black"

def cr(v):  # road flood
    if "COMPLETELY" in str(v): return "background:#7b0000;color:white;font-weight:bold"
    if "HEAVILY"    in str(v): return "background:#c0392b;color:white"
    if "PARTIALLY"  in str(v): return "background:#e67e22;color:black"
    if "MARGINALLY" in str(v): return "background:#f9e79f;color:black"
    return "background:#f8f9fa;color:#999"

def fmt(df):
    return {c:"{:.3f}" for c in df.select_dtypes("float").columns}

def highlight_total(row):
    if str(row.get("Sl No","")) == "TOTAL":
        return ["background:#1a2a3a;color:white;font-weight:bold"]*len(row)
    return [""]*len(row)

# ─── TABLE 1: Annexure 3A ────────────────────────────────────────────────────
if not sett_ply_smooth.empty:
    for ann, lbl, hc in [(ann3A_pip,"PIPING","#1a5276"),
                         (ann3A_ovt,"OVERTOPPING","#145a32")]:
        if ann.empty: continue
        display(HTML(f"<h3 style='color:{hc}'>📍 Annexure 3A — Location Hazard | {lbl}</h3>"))
        display(HTML("<p style='font-size:12px;color:#555'>"
                     "Sorted by distance from dam · "
                     "P1=IMMEDIATE (≤2h) · P2=URGENT (2–4h) · P3=HIGH (4–6h) · "
                     "P4=MODERATE (6–8h) · P5=PLANNED (8+h) — "
                     "near-dam villages correctly get P1 (least evacuation time)</p>"))
        th_l = [{"selector":"th","props":[("background",hc),("color","white"),
                 ("font-size","11px"),("padding","5px 8px"),("white-space","nowrap")]},
                {"selector":"td","props":[("font-size","11px"),("padding","3px 7px")]},
                {"selector":"tr:hover td","props":[("background","#f0f7ff")]}]
        dc = [c for c in ann.columns if "Depth" in c]
        hc_col = [c for c in ann.columns if c == "Hazard Class"]
        ec = [c for c in ann.columns if c == "Priority Code"]
        el = [c for c in ann.columns if c == "Priority Label"]
        fc = [c for c in ann.columns if c == "Flood Status"]
        styled = (ann.style
                  .applymap(cd,  subset=dc)
                  .applymap(ch,  subset=hc_col)
                  .applymap(ce,  subset=ec)
                  .applymap(cel, subset=el)
                  .applymap(cs,  subset=fc)
                  .apply(highlight_total, axis=1)
                  .format(fmt(ann))
                  .set_table_styles(th_l)
                  .set_caption(f"Annexure 3A — {DAM_NAME} {lbl} | {len(ann)} villages sorted by dam distance"))
        display(styled)
        display(HTML("<br>"))

# ─── TABLE 2: Annexure 3B PAR ─────────────────────────────────────────────────
if not sett_ply_smooth.empty:
    for ann, lbl, hc in [(ann3B_pip,"PIPING","#7b241c"),
                         (ann3B_ovt,"OVERTOPPING","#186a3b")]:
        if ann.empty: continue
        display(HTML(f"<h3 style='color:{hc}'>👥 Annexure 3B — Population At Risk | {lbl}</h3>"))
        th_l = [{"selector":"th","props":[("background",hc),("color","white"),
                 ("font-size","11px"),("padding","5px 8px"),("white-space","nowrap")]},
                {"selector":"td","props":[("font-size","11px"),("padding","3px 7px")]},
                {"selector":"tr:hover td","props":[("background","#fff9f0")]}]
        dc = [c for c in ann.columns if "Depth" in c]
        hc_col = [c for c in ann.columns if c == "Hazard Class"]
        el = [c for c in ann.columns if c == "Evacuation Priority"]
        fc = [c for c in ann.columns if c == "Flood Status Callout"]
        styled = (ann.style
                  .apply(highlight_total, axis=1)
                  .applymap(cd,  subset=dc)
                  .applymap(ch,  subset=hc_col)
                  .applymap(cel, subset=el)
                  .applymap(cs,  subset=fc)
                  .format(fmt(ann))
                  .set_table_styles(th_l)
                  .set_caption(f"Annexure 3B — {DAM_NAME} {lbl} PAR | {len(ann)-1} villages"))
        display(styled)
        display(HTML("<br>"))

# ─── TABLE 3: Zonal Stats ────────────────────────────────────────────────────
if not sett_ply_smooth.empty:
    display(HTML("<h3 style='color:#1a2a3a'>📐 Zonal Statistics — Depth · Velocity · WSE · Arrival Time</h3>"))
    ZCOLS = {
        NAME_COLUMN:   "Village",
        "pip_dist_km": "Dist.Dam(km)",
        "pip_dep_max": "Pip Dep.Max(m)",   "pip_dep_avg": "Pip Dep.Avg(m)",
        "pip_vel_max": "Pip Vel.Max(m/s)", "pip_vel_avg": "Pip Vel.Avg(m/s)",
        "pip_wse_max": "Pip WSE.Max(m)",
        "pip_arr_min": "Pip Arr.Min(h)",   "pip_arr_avg": "Pip Arr.Avg(h)",
        "ovt_dep_max": "Ovt Dep.Max(m)",   "ovt_dep_avg": "Ovt Dep.Avg(m)",
        "ovt_vel_max": "Ovt Vel.Max(m/s)", "ovt_vel_avg": "Ovt Vel.Avg(m/s)",
        "ovt_wse_max": "Ovt WSE.Max(m)",
        "ovt_arr_min": "Ovt Arr.Min(h)",   "ovt_arr_avg": "Ovt Arr.Avg(h)",
    }
    use = [c for c in ZCOLS if c in ply_pip.columns]
    df_z = ply_pip[ply_pip["inundated"]][use].rename(columns=ZCOLS).copy()
    if "Dist.Dam(km)" in df_z.columns:
        df_z = df_z.sort_values("Dist.Dam(km)")
    df_z.to_csv(os.path.join(OUT_DIR, "Zonal_Statistics.csv"), index=False)
    dep_c = [v for k,v in ZCOLS.items() if "dep" in k and v in df_z.columns]
    display(df_z.style
            .applymap(cd, subset=dep_c)
            .format(fmt(df_z))
            .set_table_styles(TH)
            .set_caption(f"Zonal Statistics — {len(df_z)} inundated settlements"))
    display(HTML("<br>"))

# ─── TABLE 4: Roads ──────────────────────────────────────────────────────────
if not roads_final.empty:
    display(HTML("<h3 style='color:#2e4057'>🛣️ Road Inundation — Individual Segments</h3>"))
    flooded = roads_final[roads_final["pip_inundated"] | roads_final["ovt_inundated"]].copy()
    if "dist_dam_km" in flooded.columns:
        flooded = flooded.sort_values("dist_dam_km")
    RDCOLS = {"name":"Road Name","road_cat":"Category","total_km":"Total(km)",
              "pip_inund_km":"Pip Inund(km)","pip_inund_pct":"Pip Inund(%)","pip_flood_st":"Pip Status",
              "pip_dep_max":"Pip Depth(m)","pip_arr_min":"Pip Arrival(h)",
              "ovt_inund_km":"Ovt Inund(km)","ovt_inund_pct":"Ovt Inund(%)","ovt_flood_st":"Ovt Status",
              "ovt_dep_max":"Ovt Depth(m)","ovt_arr_min":"Ovt Arrival(h)",
              "dist_dam_km":"Dist.Dam(km)"}
    use = [c for c in RDCOLS if c in flooded.columns]
    df_rd = flooded[use].rename(columns=RDCOLS).copy()
    for c in df_rd.select_dtypes("object").columns:
        df_rd[c] = df_rd[c].fillna("—")
    df_rd.to_csv(os.path.join(OUT_DIR, "Roads_Inundation.csv"), index=False)
    ps = [v for k,v in RDCOLS.items() if "pip_flood_st" in k and v in df_rd.columns]
    os_ = [v for k,v in RDCOLS.items() if "ovt_flood_st" in k and v in df_rd.columns]
    display(df_rd.style
            .applymap(cr, subset=ps+os_)
            .format(fmt(df_rd))
            .set_table_styles(TH)
            .set_caption(f"Flooded Road Segments — {len(df_rd)} of {len(roads_final)} total"))

    # Road type summary
    display(HTML("<h3 style='color:#2e4057'>🛣️ Road Type Summary</h3>"))
    rd_s = roads_final.groupby("road_cat").agg(
        Segments=("osm_id","count"), Total_km=("total_km","sum"),
        Pip_Inund_km=("pip_inund_km","sum"), Ovt_Inund_km=("ovt_inund_km","sum")
    ).reset_index()
    rd_s.columns = ["Road Category","Segments","Total_km","Pip_Inund_km","Ovt_Inund_km"]
    rd_s["Pip_%"] = (rd_s["Pip_Inund_km"] / rd_s["Total_km"] * 100).round(1)
    rd_s["Ovt_%"] = (rd_s["Ovt_Inund_km"] / rd_s["Total_km"] * 100).round(1)
    rd_s = rd_s.sort_values("Total_km", ascending=False)
    rd_s.to_csv(os.path.join(OUT_DIR, "Road_Type_Summary.csv"), index=False)
    display(rd_s.style
            .background_gradient(subset=["Pip_%","Ovt_%"], cmap="Reds")
            .format(fmt(rd_s))
            .set_table_styles(TH)
            .set_caption("Road Category — Total km vs Inundated km"))
    display(HTML("<br>"))

# ─── TABLE 5: OSM POI ─────────────────────────────────────────────────────────
if not poi_final.empty:
    display(HTML("<h3 style='color:#4a235a'>🏛️ OSM Public Facilities — Flood Status & Dam Distance</h3>"))
    display(HTML("<p style='font-size:12px;color:#555'>"
                 "Sorted by distance from dam. Empty names replaced with "
                 "category+OSM ID (common in rural Maharashtra where features "
                 "exist but are unnamed in OSM).</p>"))
    POICOLS = {"category":"Category","name":"Facility Name","name_mr":"Name (Marathi)",
               "amenity":"Amenity","operator":"Operator","phone":"Phone","addr":"Address",
               "dist_dam_km":"Dist.Dam(km)",
               "pip_inund":"Pip Flooded?","pip_dep_max":"Pip Depth(m)","pip_arr_min":"Pip Arrival(h)",
               "pip_ev_cod":"Pip Evac","pip_ev_win":"Pip Evac Window",
               "ovt_inund":"Ovt Flooded?","ovt_dep_max":"Ovt Depth(m)","ovt_arr_min":"Ovt Arrival(h)",
               "ovt_ev_cod":"Ovt Evac","ovt_ev_win":"Ovt Evac Window"}
    use  = [c for c in POICOLS if c in poi_final.columns]
    df_p = poi_final.sort_values("dist_dam_km")[use].rename(columns=POICOLS).copy()
    for c in df_p.select_dtypes("object").columns:
        df_p[c] = df_p[c].replace({"":"—","nan":"—","None":"—"}).fillna("—")
    df_p.to_csv(os.path.join(OUT_DIR, "OSM_POI_Flood_Status.csv"), index=False)
    ic = [v for k,v in POICOLS.items() if "inund" in k and v in df_p.columns]
    dc_p = [v for k,v in POICOLS.items() if "dep" in k and v in df_p.columns]
    ec_p = [v for k,v in POICOLS.items() if "ev_cod" in k and v in df_p.columns]
    display(df_p.style
            .applymap(ci, subset=ic)
            .applymap(cd, subset=dc_p)
            .applymap(ce, subset=ec_p)
            .format(fmt(df_p))
            .set_table_styles(TH)
            .set_caption(f"OSM Facilities — {len(df_p)} features sorted by dam distance"))

    # POI category summary
    display(HTML("<h3 style='color:#4a235a'>📊 Facility Category Summary</h3>"))
    agg_d = {"Total":("category","count")}
    for col, nm in [("pip_inund","Pip_Flooded"),("ovt_inund","Ovt_Flooded"),
                    ("pip_dep_max","Pip_MaxDepth"),("ovt_dep_max","Ovt_MaxDepth"),
                    ("pip_arr_min","Pip_MinArrival"),("dist_dam_km","Nearest_km")]:
        if col in poi_final.columns:
            fn = "sum" if "inund" in col else ("max" if "dep" in col else "min")
            agg_d[nm] = (col, fn)
    ps = poi_final.groupby("category").agg(**agg_d).reset_index()
    ps.columns = ["Category"] + list(ps.columns[1:])
    if "Nearest_km" in ps.columns:
        ps = ps.sort_values("Nearest_km")
    ps.to_csv(os.path.join(OUT_DIR, "OSM_POI_Category_Summary.csv"), index=False)
    gc = [c for c in ps.columns if "Flooded" in c or "Depth" in c]
    display(ps.style
            .background_gradient(subset=gc, cmap="Reds")
            .format(fmt(ps))
            .set_table_styles(TH)
            .set_caption("POI Category Summary — Flooded count, max depth, earliest arrival"))
    display(HTML("<br>"))

,Sl No,Evacuation Window,Priority Order,Priority Code,Priority Label,Village Name,District,Dist from Jawalgaon Dam (km),Max Depth (m),Avg Depth (m),Max Velocity (m/s),Avg Velocity (m/s),Max WSE (m),Flood Arrival Time (hrs),Hazard Class,Hazard Level,Flood Status,Village Surrounded,Road Access Cut,Nearest Shelter,Shelter Dist (km),Vulnerability Index,Vulnerability Category,Scenario
0,1,2–4 hours,2,P2,URGENT,Hattij,Solapur,2.750,3.934,1.413,1.491,0.721,490.864,3.470,H1,Very Low,VILLAGE PARTIALLY FLOODED,NO,NO,None,2.368,0.983,Critical,Piping
1,2,2–4 hours,2,P2,URGENT,Hattij,Solapur,2.990,4.721,1.554,1.263,0.339,490.537,3.540,H1,Very Low,VILLAGE PARTIALLY FLOODED,NO,NO,Shelter for Hatij Flood Afffected People,2.008,0.688,Very High,Piping
2,3,4–6 hours,3,P3,HIGH,Hingani(ratanjan),Solapur,5.590,1.301,0.550,0.343,0.183,487.380,4.160,H1,Very Low,VILLAGE MARGINALLY AFFECTED,NO,NO,None,0.556,0.582,High,Piping
3,4,4–6 hours,3,P3,HIGH,Dhamgaon(Dumala),Solapur,8.634,1.762,0.913,0.339,0.212,482.575,5.160,H1,Very Low,VILLAGE MARGINALLY AFFECTED,NO,NO,None,0.099,0.477,High,Piping
4,5,4–6 hours,3,P3,HIGH,Raleras,Solapur,12.043,1.906,0.700,0.870,0.211,477.646,5.960,H1,Very Low,VILLAGE PARTIALLY FLOODED,NO,NO,None,0.813,0.715,Very High,Piping
5,6,6–8 hours,4,P4,MODERATE,Sasure,Solapur,14.065,2.998,1.181,1.702,0.398,475.847,6.490,H1,Very Low,VILLAGE PARTIALLY FLOODED,NO,NO,None,0.549,0.632,Very High,Piping
6,7,6–8 hours,4,P4,MODERATE,Dahithane,Solapur,17.524,4.924,1.776,1.281,0.634,470.734,7.650,H1,Very Low,VILLAGE PARTIALLY FLOODED,NO,NO,None,1.215,0.756,Very High,Piping
7,8,8+ hours,5,P5,PLANNED,Mungshi,Solapur,20.643,5.711,1.370,1.522,0.495,467.244,8.530,H1,Very Low,VILLAGE PARTIALLY FLOODED,NO,NO,None,1.307,0.678,Very High,Piping
8,9,8+ hours,5,P5,PLANNED,Waluj,Solapur,24.718,2.314,0.863,0.592,0.310,461.559,9.410,H1,Very Low,VILLAGE MARGINALLY AFFECTED,NO,NO,None,1.019,0.368,Medium,Piping


,Sl No,Evacuation Window,Priority Order,Priority Code,Priority Label,Village Name,District,Dist from Jawalgaon Dam (km),Max Depth (m),Avg Depth (m),Max Velocity (m/s),Avg Velocity (m/s),Max WSE (m),Flood Arrival Time (hrs),Hazard Class,Hazard Level,Flood Status,Village Surrounded,Road Access Cut,Nearest Shelter,Shelter Dist (km),Vulnerability Index,Vulnerability Category,Scenario
0,1,0 to 2 hour,1,P1,IMMEDIATE,Hattij,Solapur,2.750,6.095,2.114,2.809,1.049,493.010,1.170,H6,Extreme,VILLAGE MODERATELY FLOODED,NO,NO,None,2.368,0.982,Critical,Overtopping
1,2,0 to 2 hour,1,P1,IMMEDIATE,Hattij,Solapur,2.990,6.753,1.964,2.766,0.523,492.952,1.180,H1,Very Low,VILLAGE PARTIALLY FLOODED,NO,NO,Shelter for Hatij Flood Afffected People,2.008,0.414,High,Overtopping
2,3,0 to 2 hour,1,P1,IMMEDIATE,Hingani(ratanjan),Solapur,5.590,3.316,1.552,0.725,0.417,489.421,1.400,H4,High,VILLAGE PARTIALLY FLOODED,NO,NO,None,0.556,0.508,High,Overtopping
3,4,0 to 2 hour,1,P1,IMMEDIATE,Dhamgaon(Dumala),Solapur,8.634,3.615,1.533,0.720,0.197,484.434,1.900,H1,Very Low,VILLAGE MARGINALLY AFFECTED,NO,NO,None,0.099,0.186,Low,Overtopping
4,5,2–4 hours,2,P2,URGENT,Raleras,Solapur,12.043,4.073,1.616,1.703,0.458,479.923,2.310,H1,Very Low,VILLAGE MODERATELY FLOODED,NO,NO,None,0.813,0.564,High,Overtopping
5,6,2–4 hours,2,P2,URGENT,Sasure,Solapur,14.065,4.990,1.472,3.393,0.580,478.137,2.600,H1,Very Low,VILLAGE PARTIALLY FLOODED,NO,NO,None,0.549,0.429,High,Overtopping
6,7,2–4 hours,2,P2,URGENT,Dahithane,Solapur,17.524,6.771,1.963,2.763,0.882,473.071,3.300,H1,Very Low,VILLAGE MODERATELY FLOODED,NO,NO,None,1.215,0.480,High,Overtopping
7,8,2–4 hours,2,P2,URGENT,Mungshi,Solapur,20.643,7.881,1.884,2.851,0.611,469.278,3.770,H1,Very Low,VILLAGE MODERATELY FLOODED,NO,NO,None,1.307,0.411,High,Overtopping
8,9,4–6 hours,3,P3,HIGH,Waluj,Solapur,24.320,0.481,0.252,0.080,0.050,464.055,4.240,H1,Very Low,VILLAGE MARGINALLY AFFECTED,NO,NO,None,0.844,0.068,Low,Overtopping
9,10,4–6 hours,3,P3,HIGH,Waluj,Solapur,24.718,4.807,1.011,1.674,0.463,464.052,4.230,H1,Very Low,VILLAGE PARTIALLY FLOODED,NO,NO,None,1.019,0.183,Low,Overtopping


,Sl No,Village Name,District,Area Inundated (%),Original Area (ha),Inundated Area (ha),Existing Population,Effected Population (PAR),PAR % of Village,Max Depth (m),Avg Depth (m),Max Velocity (m/s),Max WSE (m),Flood Arrival (hrs),Hazard Class,Hazard Level,Evacuation Window,Evacuation Priority,Flood Status Callout,Dist from Jawalgaon Dam (km),Nearest Shelter,Shelter Dist (km),Vulnerability Index,Vulnerability Category,Scenario
0,1,Hattij,Solapur,17.0%,5.500,0.940,1751,298,17.0%,3.934,1.413,1.491,490.864,3.470,H1,Very Low,2–4 hours,URGENT,VILLAGE PARTIALLY FLOODED,2.750,None,2.368,0.983,Critical,Piping
1,2,Hattij,Solapur,7.0%,18.250,1.280,1751,123,7.0%,4.721,1.554,1.263,490.537,3.540,H1,Very Low,2–4 hours,URGENT,VILLAGE PARTIALLY FLOODED,2.990,Shelter for Hatij Flood Afffected People,2.008,0.688,Very High,Piping
2,3,Hingani(ratanjan),Solapur,4.8%,7.970,0.380,981,47,4.8%,1.301,0.550,0.343,487.380,4.160,H1,Very Low,4–6 hours,HIGH,VILLAGE MARGINALLY AFFECTED,5.590,None,0.556,0.582,High,Piping
3,4,Dhamgaon(Dumala),Solapur,1.4%,32.170,0.450,506,7,1.4%,1.762,0.913,0.339,482.575,5.160,H1,Very Low,4–6 hours,HIGH,VILLAGE MARGINALLY AFFECTED,8.634,None,0.099,0.477,High,Piping
4,5,Raleras,Solapur,8.7%,8.300,0.720,2316,201,8.7%,1.906,0.700,0.870,477.646,5.960,H1,Very Low,4–6 hours,HIGH,VILLAGE PARTIALLY FLOODED,12.043,None,0.813,0.715,Very High,Piping
5,6,Sasure,Solapur,5.3%,44.620,2.390,3427,183,5.3%,2.998,1.181,1.702,475.847,6.490,H1,Very Low,6–8 hours,MODERATE,VILLAGE PARTIALLY FLOODED,14.065,None,0.549,0.632,Very High,Piping
6,7,Dahithane,Solapur,12.3%,17.460,2.150,1765,217,12.3%,4.924,1.776,1.281,470.734,7.650,H1,Very Low,6–8 hours,MODERATE,VILLAGE PARTIALLY FLOODED,17.524,None,1.215,0.756,Very High,Piping
7,8,Mungshi,Solapur,11.5%,30.160,3.480,1291,149,11.5%,5.711,1.370,1.522,467.244,8.530,H1,Very Low,8+ hours,PLANNED,VILLAGE PARTIALLY FLOODED,20.643,None,1.307,0.678,Very High,Piping
8,9,Waluj,Solapur,1.0%,19.240,0.200,0,0,N/A,2.314,0.863,0.592,461.559,9.410,H1,Very Low,8+ hours,PLANNED,VILLAGE MARGINALLY AFFECTED,24.718,None,1.019,0.368,Medium,Piping
9,TOTAL,ALL 9 AFFECTED VILLAGES,nan,nan,nan,nan,13788,1225,8.9%,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,Piping


,Sl No,Village Name,District,Area Inundated (%),Original Area (ha),Inundated Area (ha),Existing Population,Effected Population (PAR),PAR % of Village,Max Depth (m),Avg Depth (m),Max Velocity (m/s),Max WSE (m),Flood Arrival (hrs),Hazard Class,Hazard Level,Evacuation Window,Evacuation Priority,Flood Status Callout,Dist from Jawalgaon Dam (km),Nearest Shelter,Shelter Dist (km),Vulnerability Index,Vulnerability Category,Scenario
0,1,Hattij,Solapur,42.1%,5.500,2.320,1751,737,42.1%,6.095,2.114,2.809,493.010,1.170,H6,Extreme,0 to 2 hour,IMMEDIATE,VILLAGE MODERATELY FLOODED,2.750,None,2.368,0.982,Critical,Overtopping
1,2,Hattij,Solapur,18.9%,18.250,3.450,1751,331,18.9%,6.753,1.964,2.766,492.952,1.180,H1,Very Low,0 to 2 hour,IMMEDIATE,VILLAGE PARTIALLY FLOODED,2.990,Shelter for Hatij Flood Afffected People,2.008,0.414,High,Overtopping
2,3,Hingani(ratanjan),Solapur,16.2%,7.970,1.300,981,159,16.2%,3.316,1.552,0.725,489.421,1.400,H4,High,0 to 2 hour,IMMEDIATE,VILLAGE PARTIALLY FLOODED,5.590,None,0.556,0.508,High,Overtopping
3,4,Dhamgaon(Dumala),Solapur,3.6%,32.170,1.140,506,18,3.6%,3.615,1.533,0.720,484.434,1.900,H1,Very Low,0 to 2 hour,IMMEDIATE,VILLAGE MARGINALLY AFFECTED,8.634,None,0.099,0.186,Low,Overtopping
4,5,Raleras,Solapur,32.0%,8.300,2.660,2316,742,32.0%,4.073,1.616,1.703,479.923,2.310,H1,Very Low,2–4 hours,URGENT,VILLAGE MODERATELY FLOODED,12.043,None,0.813,0.564,High,Overtopping
5,6,Sasure,Solapur,18.7%,44.620,8.340,3427,641,18.7%,4.990,1.472,3.393,478.137,2.600,H1,Very Low,2–4 hours,URGENT,VILLAGE PARTIALLY FLOODED,14.065,None,0.549,0.429,High,Overtopping
6,7,Dahithane,Solapur,31.8%,17.460,5.560,1765,562,31.8%,6.771,1.963,2.763,473.071,3.300,H1,Very Low,2–4 hours,URGENT,VILLAGE MODERATELY FLOODED,17.524,None,1.215,0.480,High,Overtopping
7,8,Mungshi,Solapur,30.9%,30.160,9.310,1291,399,30.9%,7.881,1.884,2.851,469.278,3.770,H1,Very Low,2–4 hours,URGENT,VILLAGE MODERATELY FLOODED,20.643,None,1.307,0.411,High,Overtopping
8,9,Waluj,Solapur,0.6%,22.640,0.140,1426,9,0.6%,0.481,0.252,0.080,464.055,4.240,H1,Very Low,4–6 hours,HIGH,VILLAGE MARGINALLY AFFECTED,24.320,None,0.844,0.068,Low,Overtopping
9,10,Waluj,Solapur,17.5%,19.240,3.370,0,0,N/A,4.807,1.011,1.674,464.052,4.230,H1,Very Low,4–6 hours,HIGH,VILLAGE PARTIALLY FLOODED,24.718,None,1.019,0.183,Low,Overtopping


,Village,Dist.Dam(km),Pip Dep.Max(m),Pip Dep.Avg(m),Pip Vel.Max(m/s),Pip Vel.Avg(m/s),Pip WSE.Max(m),Pip Arr.Min(h),Pip Arr.Avg(h),Ovt Dep.Max(m),Ovt Dep.Avg(m),Ovt Vel.Max(m/s),Ovt Vel.Avg(m/s),Ovt WSE.Max(m),Ovt Arr.Min(h),Ovt Arr.Avg(h)
7,Hattij,2.750,3.934,1.413,1.491,0.721,490.864,3.472,3.558,6.095,2.114,2.809,1.049,493.010,1.173,1.213
20,Hattij,2.990,4.721,1.554,1.263,0.339,490.537,3.542,3.604,6.753,1.964,2.766,0.523,492.952,1.176,1.237
13,Hingani(ratanjan),5.590,1.301,0.550,0.343,0.183,487.380,4.164,4.170,3.316,1.552,0.725,0.417,489.421,1.404,1.410
10,Dhamgaon(Dumala),8.634,1.762,0.913,0.339,0.212,482.575,5.157,5.165,3.615,1.533,0.720,0.197,484.434,1.903,1.909
18,Raleras,12.043,1.906,0.700,0.870,0.211,477.646,5.962,5.978,4.073,1.616,1.703,0.458,479.923,2.308,2.359
2,Sasure,14.065,2.998,1.181,1.702,0.398,475.847,6.491,6.561,4.990,1.472,3.393,0.580,478.137,2.601,2.694
1,Dahithane,17.524,4.924,1.776,1.281,0.634,470.734,7.650,7.674,6.771,1.963,2.763,0.882,473.071,3.299,3.350
0,Mungshi,20.643,5.711,1.370,1.522,0.495,467.244,8.529,8.583,7.881,1.884,2.851,0.611,469.278,3.769,3.788
19,Waluj,24.718,2.314,0.863,0.592,0.310,461.559,9.414,9.418,4.807,1.011,1.674,0.463,464.052,4.232,4.250


,Road Name,Category,Total(km),Pip Inund(km),Pip Inund(%),Pip Status,Ovt Inund(km),Ovt Inund(%),Ovt Status,Dist.Dam(km)
154,Village Road — Jotibachiwadi,Village Road,0.878,0.164,18.700,ROAD MARGINALLY AFFECTED,0.166,18.900,ROAD MARGINALLY AFFECTED,0.000
158,Kutcha Track — Jotibachiwadi,Kutcha Track,0.254,0.000,0.000,NOT FLOODED,0.140,54.900,ROAD PARTIALLY FLOODED,0.000
173,Internal Road,Internal Road,0.952,0.000,0.000,NOT FLOODED,0.112,11.800,ROAD MARGINALLY AFFECTED,0.000
823,Kutcha Track — Ambegaon,Kutcha Track,0.334,0.000,0.000,NOT FLOODED,0.156,46.800,ROAD PARTIALLY FLOODED,0.000
1674,Kutcha Track — Ambegaon,Kutcha Track,1.061,0.000,0.000,NOT FLOODED,0.121,11.400,ROAD MARGINALLY AFFECTED,0.000
1927,Service Road — Jotibachiwadi,Service Road,0.301,0.000,0.000,NOT FLOODED,0.028,9.200,ROAD MARGINALLY AFFECTED,0.000
1901,Kutcha Track,Kutcha Track,0.443,0.000,0.000,NOT FLOODED,0.080,18.000,ROAD MARGINALLY AFFECTED,0.000
1162,Kutcha Track,Kutcha Track,0.511,0.000,0.000,NOT FLOODED,0.083,16.200,ROAD MARGINALLY AFFECTED,0.000
155,Village Road — Jotibachiwadi,Village Road,0.050,0.050,100.000,ROAD COMPLETELY FLOODED,0.050,100.000,ROAD COMPLETELY FLOODED,0.081
146,Village Road — Jotibachiwadi,Village Road,1.462,0.108,7.400,ROAD MARGINALLY AFFECTED,0.230,15.700,ROAD MARGINALLY AFFECTED,0.112


,Road Category,Segments,Total_km,Pip_Inund_km,Ovt_Inund_km,Pip_%,Ovt_%
10,Village Road,275,548.764,7.653,11.351,1.400,2.100
6,Other Road,9,514.970,0.500,1.337,0.100,0.300
3,Internal Road,1454,361.925,7.528,12.941,2.100,3.600
4,Kutcha Track,384,273.665,13.610,20.636,5.000,7.500
5,Major Road,74,133.164,1.275,4.465,1.000,3.400
9,State Highway,61,130.636,0.000,0.000,0.000,0.000
0,District Road,40,102.039,0.600,1.277,0.600,1.300
8,Service Road,320,56.395,1.389,2.588,2.500,4.600
2,Highway Link,16,6.063,0.000,0.000,0.000,0.000
1,Footpath,9,0.973,0.000,0.000,0.000,0.000


,Category,Facility Name,Name (Marathi),Amenity,Operator,Phone,Address,Dist.Dam(km),Pip Flooded?,Pip Depth(m),Pip Arrival(h),Pip Evac,Pip Evac Window,Ovt Flooded?,Ovt Depth(m),Ovt Arrival(h),Ovt Evac,Ovt Evac Window
0,Temple / Place of Worship,Tuljabhavani mandir,—,place_of_worship,—,—,—,0.000,False,nan,nan,P5,8+ hours,False,nan,nan,P5,8+ hours
1,Hospital,Sai Mangal Hospital,—,hospital,—,—,"Borgaonkar Complex, Naldurg Road, Above District Co-operative Central Bank, Tuljapur",0.000,False,nan,nan,P5,8+ hours,False,nan,nan,P5,8+ hours
2,Hospital,Peshwe Hospital,—,hospital,—,—,Near Water Tank Shukrawar Peth Tuljapur,0.000,False,nan,nan,P5,8+ hours,False,nan,nan,P5,8+ hours
3,Hospital,Malba Hospital,—,hospital,—,—,"Osmanabad Road, Kaman Cese, Tulajpur",0.000,False,nan,nan,P5,8+ hours,False,nan,nan,P5,8+ hours
4,Hospital,Yadav Nursing and Maternity Home - Tuljapur,—,hospital,—,—,"Mathurai Nagar,Near St Bus Stand",0.000,False,nan,nan,P5,8+ hours,False,nan,nan,P5,8+ hours
5,Primary Health Centre (PHC),PHC Gaudgaon,—,—,—,—,Gaudgaon,0.000,False,nan,nan,P5,8+ hours,False,nan,nan,P5,8+ hours
360,Water Well,Water Well,—,—,—,—,—,0.000,False,nan,nan,P5,8+ hours,False,nan,nan,P5,8+ hours
24,Water Well,Water Well,—,—,—,—,—,0.000,False,nan,nan,P5,8+ hours,False,nan,nan,P5,8+ hours
28,Water Well,Water Well,—,—,—,—,—,0.000,False,nan,nan,P5,8+ hours,False,nan,nan,P5,8+ hours
25,Water Well,Water Well,—,—,—,—,—,0.000,False,nan,nan,P5,8+ hours,False,nan,nan,P5,8+ hours


,Category,Total,Pip_Flooded,Ovt_Flooded,Pip_MaxDepth,Ovt_MaxDepth,Pip_MinArrival,Nearest_km
0,Bridge,147,19,21,7.052,8.975,2.658,0.000
1,Bus Station,1,0,0,nan,nan,nan,0.000
2,Bus Stop,4,0,0,nan,nan,nan,0.000
3,College / University,1,0,0,nan,nan,nan,0.000
4,Dam,1,0,0,nan,nan,nan,0.000
5,Electricity Substation,3,0,0,nan,nan,nan,0.000
6,Hospital,9,0,0,nan,nan,nan,0.000
7,Primary Health Centre (PHC),3,0,0,nan,nan,nan,0.000
8,School,2,0,0,nan,nan,nan,0.000
9,Temple / Place of Worship,4,0,0,nan,nan,nan,0.000


## Cell 21 — Executive Summary Report

In [ ]:
# Cell 21
line = "="*68
print(line)
print(f"  {DAM_NAME.upper()} — PAR ANALYSIS EXECUTIVE SUMMARY")
print(f"  District: {DAM_DISTRICT} | State: {DAM_STATE}")
print(line)

wgs = dam_gdf.to_crs("EPSG:4326").geometry.iloc[0]
print(f"\n  📍 Dam : Lat {wgs.y:.4f}°N | Lon {wgs.x:.4f}°E")
print(f"\n  🌊 INUNDATION EXTENTS")
print(f"     Piping      : {area_pip_ha:>10,.2f} ha  ({area_pip_ha/100:.3f} km²)")
print(f"     Overtopping : {area_ovt_ha:>10,.2f} ha  ({area_ovt_ha/100:.3f} km²)")

if not sett_ply_smooth.empty:
    for df, sc, lbl in [(ply_pip,"pip","PIPING"), (ply_ovt,"ovt","OVERTOPPING")]:
        aff     = df[df["inundated"]]
        tot_pop = int(pd.to_numeric(df.get(POP_COLUMN, pd.Series([0]*len(df))),
                                    errors="coerce").fillna(0).sum())
        tot_par = int(df.get(f"{sc}_PAR", pd.Series([0]*len(df))).fillna(0).sum())
        i_ha    = float(aff.get(f"{sc}_inund_ha", pd.Series([0]*len(aff))).fillna(0).sum())
        o_ha    = float(df.get(f"{sc}_orig_ha",   pd.Series([0]*len(df))).fillna(0).sum())
        print(f"\n  ── {lbl} {'─'*44}")
        print(f"     Villages assessed    : {len(df)}")
        print(f"     Villages inundated   : {len(aff)}")
        print(f"     Settlement area inundated : {i_ha:,.2f} ha "
              f"({i_ha/o_ha*100:.1f}% of total)" if o_ha else "")
        print(f"     Total population     : {tot_pop:,}")
        print(f"     Population At Risk   : {tot_par:,}"
              + (f"  ({tot_par/tot_pop*100:.1f}% of total)" if tot_pop else ""))
        for st in ["VILLAGE COMPLETELY FLOODED","VILLAGE SURROUNDED BY FLOOD",
                   "VILLAGE ISOLATED — ROAD ACCESS CUT","VILLAGE HEAVILY FLOODED",
                   "VILLAGE MODERATELY FLOODED","VILLAGE PARTIALLY FLOODED"]:
            col  = f"{sc}_flood_st"
            cnt  = int((df.get(col, pd.Series([""]*len(df))) == st).sum())
            par_st = int(df[df.get(col, pd.Series([""]*len(df))) == st].get(
                f"{sc}_PAR", pd.Series([0])).fillna(0).sum())
            if cnt: print(f"     {st:<44}: {cnt} | PAR: {par_st:,}")
        print(f"  EVACUATION PRIORITIES:")
        for o, w, c, l in [(1,"0–2h","P1","IMMEDIATE"),(2,"2–4h","P2","URGENT"),
                            (3,"4–6h","P3","HIGH"),(4,"6–8h","P4","MODERATE"),(5,"8+h","P5","PLANNED")]:
            cnt   = int((df.get(f"{sc}_ev_ord", pd.Series([5]*len(df))) == o).sum())
            par_e = int(df[df.get(f"{sc}_ev_ord", pd.Series([5]*len(df))) == o].get(
                f"{sc}_PAR", pd.Series([0])).fillna(0).sum())
            if cnt: print(f"     {c} {l:<12} ({w:<6}): {cnt:>3} villages | PAR: {par_e:,}")

if not roads_final.empty:
    print(f"\n  🛣️  ROADS FLOODED")
    print(f"     Piping      : {roads_final['pip_inund_km'].sum():.2f} km "
          f"({int(roads_final['pip_inundated'].sum())} segments)")
    print(f"     Overtopping : {roads_final['ovt_inund_km'].sum():.2f} km "
          f"({int(roads_final['ovt_inundated'].sum())} segments)")

if not poi_final.empty and "pip_inund" in poi_final.columns:
    print(f"\n  🏛️  OSM FACILITIES FLOODED (Piping / Overtopping)")
    for cat in sorted(poi_final["category"].unique()):
        s = poi_final[poi_final["category"] == cat]
        p = int(s["pip_inund"].sum()); o = int(s["ovt_inund"].sum())
        if p or o:
            print(f"     {cat:<35}: Pip={p}/{len(s)} | Ovt={o}/{len(s)}")

print(f"\n  📁 Output folder: {OUT_DIR}")
print(f"  📋 CSV files:"); [print(f"     → {f}")
    for f in sorted(os.listdir(OUT_DIR)) if f.endswith(".csv")]
print(line)
print("  ✅ ANALYSIS COMPLETE")
print(line)

  JAWALGAON DAM — PAR ANALYSIS EXECUTIVE SUMMARY
  District: Solapur | State: Maharashtra

  📍 Dam : Lat 18.0171°N | Lon 75.9010°E

  🌊 INUNDATION EXTENTS
     Piping      :   2,152.54 ha  (21.525 km²)
     Overtopping :   3,240.84 ha  (32.408 km²)

  ── PIPING ────────────────────────────────────────────
     Villages assessed    : 22
     Villages inundated   : 9
     Settlement area inundated : 11.98 ha (1.7% of total)
     Total population     : 69,829
     Population At Risk   : 1,225  (1.8% of total)
     VILLAGE PARTIALLY FLOODED                   : 6 | PAR: 1,171
  EVACUATION PRIORITIES:
     P2 URGENT       (2–4h  ):   2 villages | PAR: 421
     P3 HIGH         (4–6h  ):   3 villages | PAR: 255
     P4 MODERATE     (6–8h  ):   2 villages | PAR: 400
     P5 PLANNED      (8+h   ):  15 villages | PAR: 149

  ── OVERTOPPING ────────────────────────────────────────────
     Villages assessed    : 22
     Villages inundated   : 10
     Settlement area inundated : 37.60 ha (5.3% of t

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  COMPREHENSIVE DISTANCE FROM DAM + ADDITIONAL EAP OUTPUTS
#  Paste as a NEW CELL at the end of your notebook.
#  Generates:
#    1. Master Distance CSV — every feature sorted by river chainage
#    2. Alert Zone Classification (Zone I / II / III)
#    3. Critical Infrastructure At-Risk Summary
#    4. Piping vs Overtopping Comparison Table
#    5. Flood Wave Travel Time Table
#    6. Village Flood Sequence Table (order villages flood arrive)
# ══════════════════════════════════════════════════════════════════════

import gc
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display as ipy_display, HTML

# ── River centerline for chainage ─────────────────────────────────────
riv  = river.geometry.unary_union
line = max(riv.geoms, key=lambda l: l.length) if hasattr(riv, "geoms") else riv
d0   = line.project(dam_point)

def chainage_km(geom):
    """Distance (km) from dam along river centerline."""
    pt = geom if geom.geom_type == "Point" else geom.centroid
    return round(abs(line.project(pt) - d0) / 1000, 3)

# ── Alert zone classification ──────────────────────────────────────────
# Based on Bori Dam EAP standard zones
def alert_zone(dist_km, arr_hrs):
    """
    Zone I  — 0–5 km  OR arrival ≤ 2 hrs  → RED  (Immediate danger)
    Zone II — 5–15 km OR arrival ≤ 6 hrs  → ORANGE (High risk)
    Zone III— >15 km  OR arrival > 6 hrs  → YELLOW (Moderate risk)
    """
    arr = float(arr_hrs) if arr_hrs is not None else 9999
    if dist_km <= 5  or arr <= 2:  return ("Zone I",   "RED",    "IMMEDIATE DANGER")
    if dist_km <= 15 or arr <= 6:  return ("Zone II",  "ORANGE", "HIGH RISK")
    return                                ("Zone III", "YELLOW", "MODERATE RISK")

# ══════════════════════════════════════════════════════════════════════
# OUTPUT 1 — MASTER DISTANCE TABLE (all features)
# ══════════════════════════════════════════════════════════════════════

all_rows = []

# ── Settlements ────────────────────────────────────────────────────────
if not sett_ply_smooth.empty:
    for _, r in ply_pip.iterrows():
        dist   = chainage_km(r.geometry)
        arr_p  = r.get("pip_arr_min", np.nan)
        arr_o  = r.get("ovt_arr_min", np.nan)
        zone   = alert_zone(dist, arr_p if not np.isnan(float(arr_p or 9999)) else arr_o)
        pop    = int(r.get(POP_COLUMN, 0) or 0)
        par_p  = int(r.get("pip_PAR", 0) or 0)

        # Safely get ovt_PAR and inundated status for overtopping scenario
        ovt_inundated_status = False
        ovt_inund_pct = 0
        ovt_dep_max = 0
        ovt_arr_min = np.nan
        ovt_ev_win = ""
        ovt_par = 0
        ovt_hz_cls = ""
        ovt_flood_st = ""

        if r.name in ply_ovt.index:
            ovt_row = ply_ovt.loc[r.name]
            ovt_inundated_status = ovt_row.get("inundated", False)
            ovt_inund_pct = ovt_row.get("ovt_inund_pct", 0)
            ovt_dep_max = ovt_row.get("ovt_dep_max", 0)
            ovt_arr_min = ovt_row.get("ovt_arr_min", np.nan)
            ovt_ev_win = ovt_row.get("ovt_ev_win", "")
            ovt_par = ovt_row.get("ovt_PAR", 0)
            ovt_hz_cls = ovt_row.get("ovt_hz_cls", "")
            ovt_flood_st = ovt_row.get("ovt_flood_st", "")

        all_rows.append({
            "Feature Type":            "Settlement / Village",
            "Feature Name":            r.get(NAME_COLUMN, ""),
            "District":                DAM_DISTRICT,
            "Distance from Dam (km)":  dist,
            "Alert Zone":              zone[0],
            "Alert Colour":            zone[1],
            "Zone Description":        zone[2],
            "Piping — Inundated?":     "YES" if r.get("inundated") else "NO",
            "Piping — Area Flooded (%)": round(float(r.get("pip_inund_pct", 0) or 0), 1),
            "Piping — Max Depth (m)":  round(float(r.get("pip_dep_max", 0) or 0), 3),
            "Piping — Arrival (hrs)":  round(float(arr_p), 2) if not np.isnan(float(arr_p or 9999)) else "Not Reached",
            "Piping — Evac Window":    r.get("pip_ev_win", ""),
            "Piping — PAR":            par_p,
            "Overtopping — Inundated?": "YES" if ovt_inundated_status else "NO",
            "Overtopping — Area Flooded (%)": round(float(ovt_inund_pct), 1),
            "Overtopping — Max Depth (m)": round(float(ovt_dep_max), 3),
            "Overtopping — Arrival (hrs)": round(float(ovt_arr_min), 2) if not np.isnan(float(ovt_arr_min or 9999)) else "Not Reached",
            "Overtopping — Evac Window":   ovt_ev_win,
            "Overtopping — PAR":       ovt_par,
            "Population":              pop,
            "Nearest Shelter":         r.get("sh_name", ""),
            "Shelter Distance (km)":   round(float(r.get("sh_dist_km", 0) or 0), 2),
            "Hazard Class (Piping)":   r.get("pip_hz_cls", ""),
            "Hazard Class (Overtop)":  ovt_hz_cls,
            "Flood Status (Piping)":   r.get("pip_flood_st", ""),
            "Flood Status (Overtop)":  ovt_flood_st,
        })

# ── Bridges ────────────────────────────────────────────────────────────
if poi_final is not None and not poi_final.empty:
    bridges = poi_final[poi_final["category"] == "Bridge"].copy()
    for _, r in bridges.iterrows():
        dist  = float(r.get("dist_dam_km", chainage_km(r.geometry)))
        arr_p = r.get("pip_arr_min", np.nan)
        arr_o = r.get("ovt_arr_min", np.nan)
        zone  = alert_zone(dist, arr_p if not np.isnan(float(arr_p or 9999)) else arr_o)
        flooded_p = bool(r.get("pip_inund", False))
        flooded_o = bool(r.get("ovt_inund", False))
        all_rows.append({
            "Feature Type":            "Bridge",
            "Feature Name":            r.get("name", ""),
            "District":                DAM_DISTRICT,
            "Distance from Dam (km)":  round(dist, 3),
            "Alert Zone":              zone[0],
            "Alert Colour":            zone[1],
            "Zone Description":        zone[2],
            "Piping — Inundated?":     "YES" if flooded_p else "NO",
            "Piping — Max Depth (m)":  round(float(r.get("pip_dep_max", 0) or 0), 3),
            "Piping — Arrival (hrs)":  round(float(arr_p), 2) if not np.isnan(float(arr_p or 9999)) else "Not Reached",
            "Piping — Evac Window":    r.get("pip_ev_win", ""),
            "Overtopping — Inundated?": "YES" if flooded_o else "NO",
            "Overtopping — Max Depth (m)": round(float(r.get("ovt_dep_max", 0) or 0), 3),
            "Overtopping — Arrival (hrs)": round(float(arr_o), 2) if not np.isnan(float(arr_o or 9999)) else "Not Reached",
            "Overtopping — Evac Window":   r.get("ovt_ev_win", ""),
            "Remarks": "⚠️ BRIDGE SUBMERGED — ACCESS CUT" if (flooded_p or flooded_o) else "",
        })

# ── POI (schools, hospitals, temples, PHC, police etc.) ───────────────
EXCL = {"Bridge", "Water Well", "Water Tower / Storage Tank"}
if poi_final is not None and not poi_final.empty:
    poi_other = poi_final[~poi_final["category"].isin(EXCL)].copy()
    for _, r in poi_other.iterrows():
        dist  = float(r.get("dist_dam_km", chainage_km(r.geometry)))
        arr_p = r.get("pip_arr_min", np.nan)
        arr_o = r.get("ovt_arr_min", np.nan)
        zone  = alert_zone(dist, arr_p if not np.isnan(float(arr_p or 9999)) else arr_o)
        all_rows.append({
            "Feature Type":            r.get("category", "Facility"),
            "Feature Name":            r.get("name", ""),
            "District":                DAM_DISTRICT,
            "Distance from Dam (km)":  round(dist, 3),
            "Alert Zone":              zone[0],
            "Alert Colour":            zone[1],
            "Zone Description":        zone[2],
            "Piping — Inundated?":     "YES" if r.get("pip_inund") else "NO",
            "Piping — Max Depth (m)":  round(float(r.get("pip_dep_max", 0) or 0), 3),
            "Piping — Arrival (hrs)":  round(float(arr_p), 2) if not np.isnan(float(arr_p or 9999)) else "Not Reached",
            "Piping — Evac Window":    r.get("pip_ev_win", ""),
            "Overtopping — Inundated?": "YES" if r.get("ovt_inund") else "NO",
            "Overtopping — Max Depth (m)": round(float(r.get("ovt_dep_max", 0) or 0), 3),
            "Overtopping — Arrival (hrs)": round(float(arr_o), 2) if not np.isnan(float(arr_o or 9999)) else "Not Reached",
            "Overtopping — Evac Window":   r.get("ovt_ev_win", ""),
        })

# ── Roads (flooded segments only) ─────────────────────────────────────
if roads_final is not None and not roads_final.empty:
    flooded_roads = roads_final[
        roads_final["pip_inundated"] | roads_final["ovt_inundated"]
    ].copy()
    for _, r in flooded_roads.iterrows():
        dist = float(r.get("dist_dam_km", chainage_km(r.geometry)))
        zone = alert_zone(dist, None)
        all_rows.append({
            "Feature Type":            f"Road — {r.get('road_cat','Road')}",
            "Feature Name":            r.get("name", ""),
            "District":                DAM_DISTRICT,
            "Distance from Dam (km)":  round(dist, 3),
            "Alert Zone":              zone[0],
            "Alert Colour":            zone[1],
            "Zone Description":        zone[2],
            "Piping — Inundated?":     "YES" if r.get("pip_inundated") else "NO",
            "Piping — Inundated km":   round(float(r.get("pip_inund_km", 0) or 0), 3),
            "Piping — Inundated %":    round(float(r.get("pip_inund_pct", 0) or 0), 1),
            "Piping — Flood Status":   r.get("pip_flood_st", ""),
            "Overtopping — Inundated?": "YES" if r.get("ovt_inundated") else "NO",
            "Overtopping — Inundated km": round(float(r.get("ovt_inund_km", 0) or 0), 3),
            "Overtopping — Inundated %":  round(float(r.get("ovt_inund_pct", 0) or 0), 1),
            "Overtopping — Flood Status": r.get("ovt_flood_st", ""),
            "Total Road Length (km)":  round(float(r.get("total_km", 0) or 0), 3),
        })

# ── Build master DataFrame sorted by distance ─────────────────────────
df_master = pd.DataFrame(all_rows)
df_master = df_master.sort_values("Distance from Dam (km)").reset_index(drop=True)
df_master.index = df_master.index + 1  # start at row 1

# Save
master_path = os.path.join(OUT_DIR, "Master_Distance_From_Dam.csv")
df_master.to_csv(master_path)
print(f"✅ Master distance CSV saved → {os.path.basename(master_path)}")
print(f"   {len(df_master)} features | sorted by distance from {DAM_NAME}")


# ══════════════════════════════════════════════════════════════════════
# OUTPUT 2 — FLOOD WAVE TRAVEL TIME TABLE
# Shows how fast the flood front moves reach each settlement
# ══════════════════════════════════════════════════════════════════════

if not sett_ply_smooth.empty:
    travel_rows = []
    for _, r in ply_pip.iterrows():
        dist  = float(r.get("pip_dist_km", chainage_km(r.geometry)) or 0)
        arr_p = r.get("pip_arr_min", np.nan)
        arr_o = ply_ovt.loc[r.name, "ovt_arr_min"] if r.name in ply_ovt.index else np.nan

        # Wave speed = distance / arrival time
        def wave_speed(d, a):
            try:
                a = float(a)
                return round(d / a, 2) if (a > 0 and a < 9999) else None
            except: return None

        speed_p = wave_speed(dist, arr_p)
        speed_o = wave_speed(dist, arr_o)

        travel_rows.append({
            "Village":                         r.get(NAME_COLUMN, ""),
            "Distance from Dam (km)":          round(dist, 2),
            "Piping — Arrival (hrs)":          round(float(arr_p), 2) if not np.isnan(float(arr_p or 9999)) else "—",
            "Piping — Wave Speed (km/hr)":     speed_p if speed_p else "—",
            "Piping — Evac Window":            r.get("pip_ev_win", ""),
            "Piping — Priority":               r.get("pip_ev_lbl", ""),
            "Overtopping — Arrival (hrs)":     round(float(arr_o), 2) if not np.isnan(float(arr_o or 9999)) else "—",
            "Overtopping — Wave Speed (km/hr)": speed_o if speed_o else "—",
            "Overtopping — Evac Window":       ply_ovt.loc[r.name, "ovt_ev_win"] if r.name in ply_ovt.index else "",
            "Overtopping — Priority":          ply_ovt.loc[r.name, "ovt_ev_lbl"] if r.name in ply_ovt.index else "",
            "Population":                      int(r.get(POP_COLUMN, 0) or 0),
            "PAR (Piping)":                    int(r.get("pip_PAR", 0) or 0),
            "PAR (Overtopping)":               int(ply_ovt.loc[r.name, "ovt_PAR"] if r.name in ply_ovt.index else 0),
        })

    df_travel = (pd.DataFrame(travel_rows)
                 .sort_values("Distance from Dam (km)")
                 .reset_index(drop=True))
    df_travel.index += 1
    travel_path = os.path.join(OUT_DIR, "Flood_Wave_Travel_Time.csv")
    df_travel.to_csv(travel_path)
    print(f"✅ Flood wave travel time CSV → {os.path.basename(travel_path)}")


# ══════════════════════════════════════════════════════════════════════
# OUTPUT 3 — PIPING vs OVERTOPPING COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════════

if not sett_ply_smooth.empty:
    comp_rows = []
    for _, r in ply_pip.iterrows():
        vname  = r.get(NAME_COLUMN, "")
        dist   = float(r.get("pip_dist_km", 0) or 0)
        pop    = int(r.get(POP_COLUMN, 0) or 0)
        par_p  = int(r.get("pip_PAR", 0) or 0)
        par_o  = int(ply_ovt.loc[r.name, "ovt_PAR"] if r.name in ply_ovt.index else 0)
        dep_p  = float(r.get("pip_dep_max", 0) or 0)
        dep_o  = float(ply_ovt.loc[r.name, "ovt_dep_max"] if r.name in ply_ovt.index else 0)
        arr_p  = r.get("pip_arr_min", np.nan)
        arr_o  = ply_ovt.loc[r.name, "ovt_arr_min"] if r.name in ply_ovt.index else np.nan
        ipct_p = float(r.get("pip_inund_pct", 0) or 0)
        ipct_o = float(ply_ovt.loc[r.name, "ovt_inund_pct"] if r.name in ply_ovt.index else 0)

        # Worst-case scenario
        worst = "PIPING" if dep_p >= dep_o else "OVERTOPPING"
        diff_dep  = round(dep_o - dep_p, 3)
        diff_arr  = round(
            float(arr_p or 9999) - float(arr_o or 9999), 2
        )  # positive = piping arrives later

        comp_rows.append({
            "Village":                      vname,
            "Distance from Dam (km)":       round(dist, 2),
            "Population":                   pop,
            "Pip — Inund %":                f"{ipct_p:.1f}%",
            "Pip — Max Depth (m)":          round(dep_p, 3),
            "Pip — Arrival (hrs)":          round(float(arr_p), 2) if not np.isnan(float(arr_p or 9999)) else "—",
            "Pip — PAR":                    par_p,
            "Pip — Flood Status":           r.get("pip_flood_st", ""),
            "Ovt — Inund %":                f"{ipct_o:.1f}%",
            "Ovt — Max Depth (m)":          round(dep_o, 3),
            "Ovt — Arrival (hrs)":          round(float(arr_o), 2) if not np.isnan(float(arr_o or 9999)) else "—",
            "Ovt — PAR":                    par_o,
            "Ovt — Flood Status":           ply_ovt.loc[r.name, "ovt_flood_st"] if r.name in ply_ovt.index else "",
            "Worst Scenario":               worst,
            "Depth Difference (Ovt−Pip) m": diff_dep,
            "Arrival Diff (Pip−Ovt) hrs":   diff_arr,
            "Additional PAR (Ovt−Pip)":     par_o - par_p,
        })

    df_comp = (pd.DataFrame(comp_rows)
               .sort_values("Distance from Dam (km)")
               .reset_index(drop=True))
    df_comp.index += 1
    comp_path = os.path.join(OUT_DIR, "Piping_vs_Overtopping_Comparison.csv")
    df_comp.to_csv(comp_path)
    print(f"✅ Scenario comparison CSV → {os.path.basename(comp_path)}")


# ══════════════════════════════════════════════════════════════════════
# OUTPUT 4 — CRITICAL INFRASTRUCTURE AT-RISK SUMMARY
# ══════════════════════════════════════════════════════════════════════

CRITICAL_TYPES = {
    "Bridge":                       "🌉",
    "Hospital":                     "🏥",
    "Primary Health Centre (PHC)":  "🏥",
    "School":                       "🏫",
    "Police Station":               "👮",
    "Fire Station":                 "🚒",
    "Grampanchayat / Municipal Office": "🏛️",
    "Electricity Substation":       "⚡",
    "Post Office":                  "📮",
}

if poi_final is not None and not poi_final.empty:
    crit = poi_final[poi_final["category"].isin(CRITICAL_TYPES.keys())].copy()
    crit_flooded = crit[crit["pip_inund"] | crit["ovt_inund"]]

    crit_rows = []
    for _, r in crit.sort_values("dist_dam_km").iterrows():
        dist  = float(r.get("dist_dam_km", 0) or 0)
        arr_p = r.get("pip_arr_min", np.nan)
        arr_o = r.get("ovt_arr_min", np.nan)
        zone  = alert_zone(dist, arr_p if not np.isnan(float(arr_p or 9999)) else arr_o)
        crit_rows.append({
            "Icon":             CRITICAL_TYPES.get(r.get("category",""), "📍"),
            "Facility Type":    r.get("category", ""),
            "Facility Name":    r.get("name", ""),
            "Distance (km)":    round(dist, 2),
            "Alert Zone":       zone[0],
            "Pip Flooded":      "YES" if r.get("pip_inund") else "NO",
            "Ovt Flooded":      "YES" if r.get("ovt_inund") else "NO",
            "Pip Depth (m)":    round(float(r.get("pip_dep_max", 0) or 0), 2),
            "Ovt Depth (m)":    round(float(r.get("ovt_dep_max", 0) or 0), 2),
            "Pip Arrival (h)":  round(float(arr_p), 2) if not np.isnan(float(arr_p or 9999)) else "—",
            "Ovt Arrival (h)":  round(float(arr_o), 2) if not np.isnan(float(arr_o or 9999)) else "—",
            "Remarks": (
                "⚠️ FLOODED — OPERATIONS DISRUPTED" if (r.get("pip_inund") or r.get("ovt_inund"))
                else "Safe — Monitor"
            ),
        })

    df_crit = pd.DataFrame(crit_rows)
    crit_path = os.path.join(OUT_DIR, "Critical_Infrastructure_At_Risk.csv")
    df_crit.to_csv(crit_path, index=False)
    print(f"✅ Critical infrastructure CSV → {os.path.basename(crit_path)}")


# ══════════════════════════════════════════════════════════════════════
# DISPLAY — INLINE COLOUR TABLES
# ══════════════════════════════════════════════════════════════════════

TH = [
    {"selector":"th",          "props":[("background","#1a2a3a"),("color","white"),
                                         ("font-size","12px"),("padding","6px 10px"),
                                         ("white-space","nowrap")]},
    {"selector":"td",          "props":[("font-size","11px"),("padding","4px 8px"),
                                         ("border-bottom","1px solid #ddd")]},
    {"selector":"tr:hover td", "props":[("background","#f0f7ff")]},
    {"selector":"caption",     "props":[("caption-side","top"),("font-size","14px"),
                                         ("font-weight","bold"),("padding","8px 0")]},
]

def s_zone(v):
    return {"Zone I":   "background:#c0392b;color:white;font-weight:bold",
            "Zone II":  "background:#e67e22;color:white",
            "Zone III": "background:#f39c12;color:black",
    }.get(str(v), "")

def s_yn(v):
    return ("background:#c0392b;color:white;font-weight:bold"
            if str(v) == "YES" else "background:#d5f5e3;color:black")

def s_worst(v):
    return ("background:#c0392b;color:white;font-weight:bold"
            if str(v) == "OVERTOPPING" else "background:#e67e22;color:white")

def fmt(df):
    return {c:"{:.3f}" for c in df.select_dtypes("float").columns}


# ── Table 1: Master distance (first 60 rows to avoid crashing display) ─
ipy_display(HTML(
    f"<h3 style='color:#1a2a3a'>📏 Master Distance Table — All Features from {DAM_NAME}</h3>"
    f"<p style='font-size:12px;color:#555'>All {len(df_master)} features sorted by river chainage. "
    f"Full table saved to <b>Master_Distance_From_Dam.csv</b>. Showing first 60 rows.</p>"
))
show_master = df_master.head(60).copy()
zone_c  = [c for c in show_master.columns if c == "Alert Zone"]
yn_c    = [c for c in show_master.columns if "Inundated?" in c]
ipy_display(
    show_master.style
    .applymap(s_zone, subset=zone_c)
    .applymap(s_yn,   subset=yn_c)
    .format(fmt(show_master))
    .set_table_styles(TH)
    .set_caption(f"Master Distance — {DAM_NAME} | {len(df_master)} total features")
)

# ── Table 2: Flood wave travel time ────────────────────────────────────
if 'df_travel' in dir():
    ipy_display(HTML(
        "<h3 style='color:#1a5276'>🌊 Flood Wave Travel Time — Village by Village</h3>"))
    ev_p = [c for c in df_travel.columns if "Piping — Priority" in c]
    ev_o = [c for c in df_travel.columns if "Overtopping — Priority" in c]

    def s_prio(v):
        return {"IMMEDIATE":"background:#c0392b;color:white;font-weight:bold",
                "URGENT":   "background:#e67e22;color:white",
                "HIGH":     "background:#f39c12;color:black",
                "MODERATE": "background:#82e0aa;color:black",
                "PLANNED":  "background:#d5f5e3;color:black",
        }.get(str(v), "")

    ipy_display(
        df_travel.style
        .applymap(s_prio, subset=ev_p + ev_o)
        .format(fmt(df_travel))
        .set_table_styles(TH)
        .set_caption(f"Flood Wave Travel Time — {len(df_travel)} villages")
    )

# ── Table 3: Piping vs Overtopping comparison ──────────────────────────
if 'df_comp' in dir():
    ipy_display(HTML(
        "<h3 style='color:#6c3483'>⚔️ Piping vs Overtopping — Side-by-Side Comparison</h3>"))
    worst_c = [c for c in df_comp.columns if c == "Worst Scenario"]
    ipy_display(
        df_comp.style
        .applymap(s_worst, subset=worst_c)
        .format(fmt(df_comp))
        .set_table_styles(TH)
        .set_caption("Scenario Comparison — Depth / Arrival / PAR differences per village")
    )

# ── Table 4: Critical infrastructure ──────────────────────────────────
if 'df_crit' in dir() and not df_crit.empty:
    ipy_display(HTML(
        "<h3 style='color:#7b241c'>🏗️ Critical Infrastructure At Risk</h3>"))
    flooded_first = df_crit.sort_values(
        ["Pip Flooded","Ovt Flooded","Distance (km)"],
        ascending=[True, True, True])  # YES sorts after NO alphabetically — reverse:
    flooded_first = flooded_first.sort_values(
        "Distance (km)")
    zone_c2  = [c for c in flooded_first.columns if c == "Alert Zone"]
    yn_c2    = [c for c in flooded_first.columns if "Flooded" in c]
    ipy_display(
        flooded_first.style
        .applymap(s_zone, subset=zone_c2)
        .applymap(s_yn,   subset=yn_c2)
        .format(fmt(flooded_first))
        .set_table_styles(TH)
        .set_caption(f"Critical Infrastructure — {len(df_crit)} features assessed")
    )

# ══════════════════════════════════════════════════════════════════════
# FINAL FILE LIST
# ══════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print(f"  {DAM_NAME.upper()} — ALL OUTPUT FILES")
print("="*60)
for f in sorted(os.listdir(OUT_DIR)):
    if f.endswith(".csv"):
        size = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
        print(f"  📋 {f:<55} {size:>6.1f} KB")
print("="*60)

# ZIP everything
import shutil
z = shutil.make_archive("/content/Jawalgaon_PAR_FINAL", "zip", OUT_DIR)
print(f"\n✅ Download: {z}")
from google.colab import files
files.download(z)

✅ Master distance CSV saved → Master_Distance_From_Dam.csv
   403 features | sorted by distance from Jawalgaon Dam
✅ Flood wave travel time CSV → Flood_Wave_Travel_Time.csv
✅ Scenario comparison CSV → Piping_vs_Overtopping_Comparison.csv
✅ Critical infrastructure CSV → Critical_Infrastructure_At_Risk.csv


,Feature Type,Feature Name,District,Distance from Dam (km),Alert Zone,Alert Colour,Zone Description,Piping — Inundated?,Piping — Area Flooded (%),Piping — Max Depth (m),Piping — Arrival (hrs),Piping — Evac Window,Piping — PAR,Overtopping — Inundated?,Overtopping — Area Flooded (%),Overtopping — Max Depth (m),Overtopping — Arrival (hrs),Overtopping — Evac Window,Overtopping — PAR,Population,Nearest Shelter,Shelter Distance (km),Hazard Class (Piping),Hazard Class (Overtop),Flood Status (Piping),Flood Status (Overtop),Remarks,Piping — Inundated km,Piping — Inundated %,Piping — Flood Status,Overtopping — Inundated km,Overtopping — Inundated %,Overtopping — Flood Status,Total Road Length (km)
1,Bridge,Bridge,Solapur,0.000,Zone I,RED,IMMEDIATE DANGER,NO,nan,nan,Not Reached,8+ hours,nan,NO,nan,nan,Not Reached,8+ hours,nan,nan,nan,nan,nan,nan,nan,nan,,nan,nan,nan,nan,nan,nan,nan
2,Bridge,Bridge,Solapur,0.000,Zone I,RED,IMMEDIATE DANGER,NO,nan,nan,Not Reached,8+ hours,nan,NO,nan,nan,Not Reached,8+ hours,nan,nan,nan,nan,nan,nan,nan,nan,,nan,nan,nan,nan,nan,nan,nan
3,Bridge,Bridge,Solapur,0.000,Zone I,RED,IMMEDIATE DANGER,NO,nan,nan,Not Reached,8+ hours,nan,NO,nan,nan,Not Reached,8+ hours,nan,nan,nan,nan,nan,nan,nan,nan,,nan,nan,nan,nan,nan,nan,nan
4,Bridge,Bridge,Solapur,0.000,Zone I,RED,IMMEDIATE DANGER,NO,nan,nan,Not Reached,8+ hours,nan,NO,nan,nan,Not Reached,8+ hours,nan,nan,nan,nan,nan,nan,nan,nan,,nan,nan,nan,nan,nan,nan,nan
5,Bridge,Bridge,Solapur,0.000,Zone I,RED,IMMEDIATE DANGER,NO,nan,nan,Not Reached,8+ hours,nan,NO,nan,nan,Not Reached,8+ hours,nan,nan,nan,nan,nan,nan,nan,nan,,nan,nan,nan,nan,nan,nan,nan
6,Bridge,Bridge,Solapur,0.000,Zone I,RED,IMMEDIATE DANGER,NO,nan,nan,Not Reached,8+ hours,nan,NO,nan,nan,Not Reached,8+ hours,nan,nan,nan,nan,nan,nan,nan,nan,,nan,nan,nan,nan,nan,nan,nan
7,Bridge,Bridge,Solapur,0.000,Zone I,RED,IMMEDIATE DANGER,NO,nan,nan,Not Reached,8+ hours,nan,NO,nan,nan,Not Reached,8+ hours,nan,nan,nan,nan,nan,nan,nan,nan,,nan,nan,nan,nan,nan,nan,nan
8,Bridge,Bridge,Solapur,0.000,Zone I,RED,IMMEDIATE DANGER,NO,nan,nan,Not Reached,8+ hours,nan,NO,nan,nan,Not Reached,8+ hours,nan,nan,nan,nan,nan,nan,nan,nan,,nan,nan,nan,nan,nan,nan,nan
9,Bridge,Bridge,Solapur,0.000,Zone I,RED,IMMEDIATE DANGER,NO,nan,nan,Not Reached,8+ hours,nan,NO,nan,nan,Not Reached,8+ hours,nan,nan,nan,nan,nan,nan,nan,nan,,nan,nan,nan,nan,nan,nan,nan
10,Bridge,Bridge,Solapur,0.000,Zone I,RED,IMMEDIATE DANGER,NO,nan,nan,Not Reached,8+ hours,nan,NO,nan,nan,Not Reached,8+ hours,nan,nan,nan,nan,nan,nan,nan,nan,,nan,nan,nan,nan,nan,nan,nan


,Village,Distance from Dam (km),Piping — Arrival (hrs),Piping — Wave Speed (km/hr),Piping — Evac Window,Piping — Priority,Overtopping — Arrival (hrs),Overtopping — Wave Speed (km/hr),Overtopping — Evac Window,Overtopping — Priority,Population,PAR (Piping),PAR (Overtopping)
1,Ambegaon,0.000,—,—,8+ hours,PLANNED,—,—,8+ hours,PLANNED,686,0,0
2,Ambabaiwadi,0.310,—,—,8+ hours,PLANNED,—,—,8+ hours,PLANNED,1315,0,0
3,Jotibachiwadi,0.540,—,—,8+ hours,PLANNED,—,—,8+ hours,PLANNED,2276,0,0
4,Hattij,2.750,3.470000,0.790000,2–4 hours,URGENT,1.170000,2.350000,0 to 2 hour,IMMEDIATE,1751,298,737
5,Malegaon,2.850,—,—,8+ hours,PLANNED,—,—,8+ hours,PLANNED,2529,0,0
6,Hattij,2.990,3.540000,0.840000,2–4 hours,URGENT,1.180000,2.540000,0 to 2 hour,IMMEDIATE,1751,123,331
7,Ratajan,3.370,—,—,8+ hours,PLANNED,—,—,8+ hours,PLANNED,2529,0,0
8,Ladole,3.440,—,—,8+ hours,PLANNED,—,—,8+ hours,PLANNED,1254,0,0
9,Hingani(ratanjan),5.590,4.160000,1.340000,4–6 hours,HIGH,1.400000,3.980000,0 to 2 hour,IMMEDIATE,981,47,159
10,Khuttewadi,7.050,—,—,8+ hours,PLANNED,—,—,8+ hours,PLANNED,635,0,0


,Village,Distance from Dam (km),Population,Pip — Inund %,Pip — Max Depth (m),Pip — Arrival (hrs),Pip — PAR,Pip — Flood Status,Ovt — Inund %,Ovt — Max Depth (m),Ovt — Arrival (hrs),Ovt — PAR,Ovt — Flood Status,Worst Scenario,Depth Difference (Ovt−Pip) m,Arrival Diff (Pip−Ovt) hrs,Additional PAR (Ovt−Pip)
1,Ambegaon,0.000,686,0.0%,nan,—,0,NOT AFFECTED,0.0%,nan,—,0,NOT AFFECTED,OVERTOPPING,nan,nan,0
2,Ambabaiwadi,0.310,1315,0.0%,nan,—,0,NOT AFFECTED,0.0%,nan,—,0,NOT AFFECTED,OVERTOPPING,nan,nan,0
3,Jotibachiwadi,0.540,2276,0.0%,nan,—,0,NOT AFFECTED,0.0%,nan,—,0,NOT AFFECTED,OVERTOPPING,nan,nan,0
4,Hattij,2.750,1751,17.0%,3.934,3.470000,298,VILLAGE PARTIALLY FLOODED,42.1%,6.095,1.170000,737,VILLAGE MODERATELY FLOODED,OVERTOPPING,2.161,2.300,439
5,Malegaon,2.850,2529,0.0%,nan,—,0,NOT AFFECTED,0.0%,nan,—,0,NOT AFFECTED,OVERTOPPING,nan,nan,0
6,Hattij,2.990,1751,7.0%,4.721,3.540000,123,VILLAGE PARTIALLY FLOODED,18.9%,6.753,1.180000,331,VILLAGE PARTIALLY FLOODED,OVERTOPPING,2.032,2.370,208
7,Ratajan,3.370,2529,0.0%,nan,—,0,NOT AFFECTED,0.0%,nan,—,0,NOT AFFECTED,OVERTOPPING,nan,nan,0
8,Ladole,3.440,1254,0.0%,nan,—,0,NOT AFFECTED,0.0%,nan,—,0,NOT AFFECTED,OVERTOPPING,nan,nan,0
9,Hingani(ratanjan),5.590,981,4.8%,1.301,4.160000,47,VILLAGE MARGINALLY AFFECTED,16.2%,3.316,1.400000,159,VILLAGE PARTIALLY FLOODED,OVERTOPPING,2.015,2.760,112
10,Khuttewadi,7.050,635,0.0%,nan,—,0,NOT AFFECTED,0.0%,nan,—,0,NOT AFFECTED,OVERTOPPING,nan,nan,0


,Icon,Facility Type,Facility Name,Distance (km),Alert Zone,Pip Flooded,Ovt Flooded,Pip Depth (m),Ovt Depth (m),Pip Arrival (h),Ovt Arrival (h),Remarks
0,🏥,Hospital,Sai Mangal Hospital,0.000,Zone I,NO,NO,nan,nan,—,—,Safe — Monitor
1,🏥,Hospital,Peshwe Hospital,0.000,Zone I,NO,NO,nan,nan,—,—,Safe — Monitor
2,🏥,Hospital,Malba Hospital,0.000,Zone I,NO,NO,nan,nan,—,—,Safe — Monitor
3,🏥,Hospital,Yadav Nursing and Maternity Home - Tuljapur,0.000,Zone I,NO,NO,nan,nan,—,—,Safe — Monitor
4,🏥,Primary Health Centre (PHC),PHC Gaudgaon,0.000,Zone I,NO,NO,nan,nan,—,—,Safe — Monitor
5,🏥,Hospital,Kadam Hospital,0.000,Zone I,NO,NO,nan,nan,—,—,Safe — Monitor
6,🏥,Hospital,Kshirasagar Hospital,0.000,Zone I,NO,NO,nan,nan,—,—,Safe — Monitor
7,🏥,Hospital,Kutwal Multi Speciality Hospital,0.000,Zone I,NO,NO,nan,nan,—,—,Safe — Monitor
8,🌉,Bridge,Bridge,0.000,Zone I,NO,NO,nan,nan,—,—,Safe — Monitor
9,⚡,Electricity Substation,Toljapur,0.000,Zone I,NO,NO,nan,nan,—,—,Safe — Monitor



  JAWALGAON DAM — ALL OUTPUT FILES
  📋 Annexure_3A_Overtopping.csv                                2.0 KB
  📋 Annexure_3A_Piping.csv                                     1.8 KB
  📋 Annexure_3B_Overtopping_PAR.csv                            2.3 KB
  📋 Annexure_3B_Piping_PAR.csv                                 2.1 KB
  📋 Critical_Infrastructure_At_Risk.csv                       14.8 KB
  📋 Flood_Wave_Travel_Time.csv                                 2.0 KB
  📋 Master_Distance_From_Dam.csv                              70.4 KB
  📋 OSM_POI_Category_Summary.csv                               0.4 KB
  📋 OSM_POI_Flood_Status.csv                                  37.9 KB
  📋 OSM_Water_Wells.csv                                       19.4 KB
  📋 Piping_vs_Overtopping_Comparison.csv                       2.7 KB
  📋 Road_Type_Summary.csv                                      0.6 KB
  📋 Roads_Inundation.csv                                      24.3 KB
  📋 Zonal_Statistics.csv                              

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 22 — ZIP & Download

In [ ]:
# Cell 22
z = shutil.make_archive("/content/Jawalgaon_PAR_Complete", "zip", OUT_DIR)
print(f"✅ ZIP ready: {z}")
from google.colab import files
files.download(z)

✅ ZIP ready: /content/Jawalgaon_PAR_Complete.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>